In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize
import os
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [1]:
import numpy as np
import pandas as pd

In [2]:
from Model import *

In [2]:
pd.set_option("display.max_rows", 11)

In [5]:
from datetime import datetime
date = datetime.today().strftime('%m%d')
num = 1

In [6]:
asthma = "GSE240567"

In [6]:
data = pd.read_csv(f"/home/koe3/Bioinformatics/Data/Asthma/{asthma}/{asthma}_Normed_Entire_Data.csv")
x = torch.from_numpy(data.values).to(dtype = torch.float)

In [7]:
data

,DPM1,FGR,CFH,FUCA2,GCLC,NFYA,SEMA3F,CFTR,CYP51A1,RAD52,...,H4C2,H3C10,ADORA3,PIGY,H3C2,H3C3,NPBWR1,UGT1A3,UGT1A5,Sex
0,0.413739,1.117053,0.663595,0.959307,0.817929,-1.305526,0.014267,-1.029322,0.681319,0.005165,...,0.632746,0.385485,0.409453,0.418256,0.234392,0.318593,-0.503568,-1.492085,-0.873495,1
1,0.114446,2.170677,0.461724,0.591021,-2.020163,-0.397902,0.379442,-1.548556,0.913926,-0.313306,...,0.657341,0.958368,0.442234,-0.995636,1.222865,0.997287,0.258784,-0.428357,-1.576770,0
2,1.443780,1.129986,-0.535696,-0.468586,-2.786815,-0.762972,0.207087,1.027602,0.246339,1.840001,...,0.652082,0.694510,2.407472,-3.029751,0.959210,1.049761,-0.954080,1.776278,1.877062,1
3,-1.089564,0.071863,-1.647470,-0.268176,-0.097398,0.468977,0.671395,-0.511815,0.598581,-0.606560,...,0.404252,-0.110976,0.168137,-0.480853,0.024609,-0.031836,1.188082,0.572094,1.050699,0
4,0.358323,0.261149,0.664754,0.691456,-0.006163,-1.421467,0.422877,-0.179158,0.647699,-1.462079,...,-0.351118,0.102787,-0.446755,-0.083824,-0.357480,0.060902,-0.193603,0.347017,0.117870,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,-0.393063,0.563204,0.640517,1.364692,0.085180,-0.640820,0.066068,-1.513612,0.244345,-0.472774,...,0.333290,1.039323,0.406330,0.325758,1.597732,1.277206,-1.550051,-0.670255,-0.164198,0
532,0.084791,-0.307321,0.444163,-0.477689,0.647847,0.052377,0.197531,0.242334,0.533503,-1.280742,...,0.468889,0.463161,0.031190,1.450512,0.592012,0.678726,0.542118,-0.797493,-0.964889,0
533,1.185496,0.213295,0.928651,0.394510,1.295366,0.210903,-0.265704,-0.614320,-0.120929,-0.636251,...,0.138450,0.055977,-0.414633,1.724079,0.336815,0.412270,-0.193252,-0.467288,0.253874,1
534,0.230130,0.519622,0.341761,0.525099,0.611224,0.274294,0.092080,-0.153758,0.595299,-0.513446,...,0.559750,0.697311,0.089098,0.919559,0.859773,0.803443,-0.866965,-1.112102,-0.976648,0


In [8]:
x, x.shape

(tensor([[ 0.4137,  1.1171,  0.6636,  ..., -1.4921, -0.8735,  1.0000],
         [ 0.1144,  2.1707,  0.4617,  ..., -0.4284, -1.5768,  0.0000],
         [ 1.4438,  1.1300, -0.5357,  ...,  1.7763,  1.8771,  1.0000],
         ...,
         [ 1.1855,  0.2133,  0.9287,  ..., -0.4673,  0.2539,  1.0000],
         [ 0.2301,  0.5196,  0.3418,  ..., -1.1121, -0.9766,  0.0000],
         [-0.7269, -0.1076,  1.0011,  ..., -0.4520, -1.3123,  0.0000]]),
 torch.Size([536, 4501]))

In [9]:
male_indices, female_indices = (x[:, -1] == 1), (x[:, -1] == 0)

In [10]:
label = pd.read_csv(f"/home/koe3/Bioinformatics/Data/Asthma/{asthma}/{asthma}_Entire_Label.csv")
label

,Label
0,1
1,1
2,1
3,1
4,1
...,...
531,0
532,0
533,0
534,0


In [11]:
def fixed_s_mask(w, idx):
    '''
    Input: 
        w: weight matrix.
        idx: the indices of having values (or connections).
    Output:
        returns the weight matrix that has been forced the connections.
    '''
    sp_w = torch.sparse_coo_tensor(idx, w[idx[0], idx[1]], w.size())
    
    return sp_w.to_dense()


In [12]:
experiment = 1

In [13]:
pathway_list = pd.read_csv("/home/koe3/Bioinformatics/Data/pathway_list.txt", header=None).values.ravel()

In [14]:
from scipy import sparse
def load_sparse_indices(path):
    coo = sparse.load_npz(path)
    indices = np.vstack((coo.row, coo.col))
    return indices

In [15]:
sparse_indices = load_sparse_indices(f"/home/koe3/Bioinformatics/Data/Asthma/{asthma}/{asthma}_Gene_Pathway_Mask.npz")

In [16]:
def get_num_nodes(data_name):
    """Returns the number of input features (genes) for a given dataset."""
    node_map = {
        "LIHC": 4781, "STAD": 4790, "LUAD": 4786,
        "LUSC": 4787, "LGG": 4789, "GSE240567": 4500
    }
    if data_name not in node_map:
        raise ValueError(f"Unknown dataset name for get_num_nodes: {data_name}")
        
    return node_map[data_name]

---

## CL Part

In [17]:
model_ver="Retune_v65"
model_date="0806"
model_num=1

In [18]:
x.shape[0], male_indices.sum(), female_indices.sum()

(536, tensor(206), tensor(330))

In [19]:
data.iloc[:, -1].sum(), (data.iloc[:, -1] == 0).sum()

(206, 330)

In [20]:
sum_cmn_pathwy_node = torch.zeros((x.shape[0], 395))
sum_cmn_repr_node = torch.zeros((x.shape[0], 128))
sum_cmn_hid_node = torch.zeros((x.shape[0], 128))
sum_cmn_proj_node = torch.zeros((x.shape[0], 64))

In [21]:
sum_male_pathwy_node = torch.zeros((data.iloc[:, -1].sum(), 395))
sum_male_repr_node = torch.zeros((data.iloc[:, -1].sum(), 128))
sum_male_hid_node = torch.zeros((data.iloc[:, -1].sum(), 128))
sum_male_proj_node = torch.zeros((data.iloc[:, -1].sum(), 64))

In [22]:
sum_female_pathwy_node = torch.zeros(((data.iloc[:, -1] == 0).sum(), 395))
sum_female_repr_node = torch.zeros(((data.iloc[:, -1] == 0).sum(), 128))
sum_female_hid_node = torch.zeros(((data.iloc[:, -1] == 0).sum(), 128))
sum_female_proj_node = torch.zeros(((data.iloc[:, -1] == 0).sum(), 64))

In [23]:
sum_cmn_w1 = torch.zeros((395, x.shape[1] - 1))
sum_cmn_w2 = torch.zeros((128, 395))
sum_cmn_w3 = torch.zeros((128, 128))
sum_cmn_w4 = torch.zeros((64, 128))

In [24]:
sum_male_w1 = torch.zeros((395, x.shape[1] - 1))
sum_male_w2 = torch.zeros((128, 395))
sum_male_w3 = torch.zeros((128, 128))
sum_male_w4 = torch.zeros((64, 128))

In [25]:
sum_female_w1 = torch.zeros((395, x.shape[1] - 1))
sum_female_w2 = torch.zeros((128, 395))
sum_female_w3 = torch.zeros((128, 128))
sum_female_w4 = torch.zeros((64, 128))

In [26]:
avg_cmn_pathwy_node, avg_cmn_repr_node, avg_cmn_hid_node, avg_cmn_proj_node = [], [], [], []
avg_male_pathwy_node, avg_male_repr_node, avg_male_hid_node, avg_male_proj_node = [], [], [], []
avg_female_pathwy_node, avg_female_repr_node, avg_female_hid_node, avg_female_proj_node = [], [], [], []
avg_cmn_w1, avg_cmn_w2, avg_cmn_w3, avg_cmn_w4 = [], [], [], []
avg_male_w1, avg_male_w2, avg_male_w3, avg_male_w4 = [], [], [], []
avg_female_w1, avg_female_w2, avg_female_w3, avg_female_w4 = [], [], [], []
for experiment in range(1, 11):
    print("######################## %d experiment ########################" % (experiment))
    cl_net_hparams = [get_num_nodes(asthma), [395, 128, 64], 1, "he_normal", "relu", 0.10070817185386628] ### GSE240567
    cl_model = CLModule(cl_net_hparams, sparse_indices)
    cl_model.load_state_dict(torch.load(f"/home/koe3/Bioinformatics/Proposed/SHCL_v3/{asthma}_Result/{model_ver}/Saved_Model/[{model_date}_{model_num}]_[{experiment}]Opt_Model_StateDict.pth", map_location='cpu'))
    
    cl_model.eval()
    with torch.no_grad():
        ### common
        common_pathway_nodes = cl_model.encoder_common.encoder.act1(cl_model.encoder_common.encoder.bn1(cl_model.encoder_common.encoder.layer1(x[:, :-1])))
        common_repr_nodes = cl_model.encoder_common.encoder.act2(cl_model.encoder_common.encoder.bn2(cl_model.encoder_common.encoder.layer2(common_pathway_nodes)))
        common_head_hidden_nodes = cl_model.encoder_common.head[2](cl_model.encoder_common.head[1](cl_model.encoder_common.head[0](common_repr_nodes)))
        common_proj_embeddings = cl_model.encoder_common.head[4](common_head_hidden_nodes)
        
        sum_cmn_pathwy_node += common_pathway_nodes
        sum_cmn_repr_node += common_repr_nodes
        sum_cmn_hid_node += common_head_hidden_nodes
        sum_cmn_proj_node += common_proj_embeddings
        ### male
        male_pathway_nodes = cl_model.encoder_male.encoder.act1(cl_model.encoder_male.encoder.bn1(cl_model.encoder_male.encoder.layer1(x[male_indices, :-1])))
        male_repr_nodes = cl_model.encoder_male.encoder.act2(cl_model.encoder_male.encoder.bn2(cl_model.encoder_male.encoder.layer2(male_pathway_nodes)))
        male_head_hidden_nodes = cl_model.encoder_male.head[2](cl_model.encoder_male.head[1](cl_model.encoder_male.head[0](male_repr_nodes)))
        male_proj_embeddings = cl_model.encoder_male.head[4](male_head_hidden_nodes)
        
        sum_male_pathwy_node += male_pathway_nodes
        sum_male_repr_node += male_repr_nodes
        sum_male_hid_node += male_head_hidden_nodes
        sum_male_proj_node += male_proj_embeddings
        ### female
        female_pathway_nodes = cl_model.encoder_female.encoder.act1(cl_model.encoder_female.encoder.bn1(cl_model.encoder_female.encoder.layer1(x[female_indices, :-1])))
        female_repr_nodes = cl_model.encoder_female.encoder.act2(cl_model.encoder_female.encoder.bn2(cl_model.encoder_female.encoder.layer2(female_pathway_nodes)))
        female_head_hidden_nodes = cl_model.encoder_female.head[2](cl_model.encoder_female.head[1](cl_model.encoder_female.head[0](female_repr_nodes)))
        female_proj_embeddings = cl_model.encoder_female.head[4](female_head_hidden_nodes)
        
        sum_female_pathwy_node += female_pathway_nodes
        sum_female_repr_node += female_repr_nodes
        sum_female_hid_node += female_head_hidden_nodes
        sum_female_proj_node += female_proj_embeddings
        
        sum_cmn_w1 += cl_model.encoder_common.encoder.layer1.weight.data
        sum_cmn_w2 += cl_model.encoder_common.encoder.layer2.weight.data
        sum_cmn_w3 += cl_model.encoder_common.head[0].weight.data
        sum_cmn_w4 += cl_model.encoder_common.head[4].weight.data
        
        sum_male_w1 += cl_model.encoder_male.encoder.layer1.weight.data
        sum_male_w2 += cl_model.encoder_male.encoder.layer2.weight.data
        sum_male_w3 += cl_model.encoder_male.head[0].weight.data
        sum_male_w4 += cl_model.encoder_male.head[4].weight.data
        
        sum_female_w1 += cl_model.encoder_female.encoder.layer1.weight.data
        sum_female_w2 += cl_model.encoder_female.encoder.layer2.weight.data
        sum_female_w3 += cl_model.encoder_female.head[0].weight.data
        sum_female_w4 += cl_model.encoder_female.head[4].weight.data
    
avg_cmn_pathwy_node = sum_cmn_pathwy_node/10
avg_cmn_repr_node = sum_cmn_repr_node/10
avg_cmn_hid_node = sum_cmn_hid_node/10
avg_cmn_proj_node = sum_cmn_proj_node/10
avg_male_pathwy_node = sum_male_pathwy_node/10
avg_male_repr_node = sum_male_repr_node/10
avg_male_hid_node = sum_male_hid_node/10
avg_male_proj_node = sum_male_proj_node/10
avg_female_pathwy_node = sum_female_pathwy_node/10
avg_female_repr_node = sum_female_repr_node/10
avg_female_hid_node = sum_female_hid_node/10
avg_female_proj_node = sum_female_proj_node/10

avg_cmn_w1 = sum_cmn_w1/10
avg_cmn_w2 = sum_cmn_w2/10
avg_cmn_w3 = sum_cmn_w3/10
avg_cmn_w4 = sum_cmn_w4/10
avg_male_w1 = sum_male_w1/10
avg_male_w2 = sum_male_w2/10
avg_male_w3 = sum_male_w3/10
avg_male_w4 = sum_male_w4/10
avg_female_w1 = sum_female_w1/10
avg_female_w2 = sum_female_w2/10
avg_female_w3 = sum_female_w3/10
avg_female_w4 = sum_female_w4/10

######################## 1 experiment ########################
######################## 2 experiment ########################
######################## 3 experiment ########################
######################## 4 experiment ########################
######################## 5 experiment ########################
######################## 6 experiment ########################
######################## 7 experiment ########################
######################## 8 experiment ########################
######################## 9 experiment ########################
######################## 10 experiment ########################


In [27]:
date

'0911'

In [28]:
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Common_Pathway_Nodes.csv", avg_cmn_pathwy_node.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Common_Representation_Nodes.csv", avg_cmn_repr_node.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Common_Head_Hidden_Nodes.csv", avg_cmn_hid_node.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Common_Projection_Nodes.csv", avg_cmn_proj_node.cpu().detach().numpy(), delimiter = ",")

In [29]:
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Male_Pathway_Nodes.csv", avg_male_pathwy_node.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Male_Representation_Nodes.csv", avg_male_repr_node.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Male_Head_Hidden_Nodes.csv", avg_male_hid_node.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Male_Projection_Nodes.csv", avg_male_proj_node.cpu().detach().numpy(), delimiter = ",")

In [30]:
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Female_Pathway_Nodes.csv", avg_female_pathwy_node.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Female_Representation_Nodes.csv", avg_female_repr_node.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Female_Head_Hidden_Nodes.csv", avg_female_hid_node.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Female_Projection_Nodes.csv", avg_female_proj_node.cpu().detach().numpy(), delimiter = ",")

In [31]:
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Common_Encoder_Layer1_Weights.csv", avg_cmn_w1.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Common_Encoder_Layer2_Weights.csv", avg_cmn_w2.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Common_Head_Layer1_Weights.csv", avg_cmn_w3.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Common_Head_Layer2_Weights.csv", avg_cmn_w4.cpu().detach().numpy(), delimiter = ",")

In [32]:
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Male_Encoder_Layer1_Weights.csv", avg_male_w1.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Male_Encoder_Layer2_Weights.csv", avg_male_w2.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Male_Head_Layer1_Weights.csv", avg_male_w3.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Male_Head_Layer2_Weights.csv", avg_male_w4.cpu().detach().numpy(), delimiter = ",")

In [33]:
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Female_Encoder_Layer1_Weights.csv", avg_female_w1.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Female_Encoder_Layer2_Weights.csv", avg_female_w2.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Female_Head_Layer1_Weights.csv", avg_female_w3.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Female_Head_Layer2_Weights.csv", avg_female_w4.cpu().detach().numpy(), delimiter = ",")

In [34]:
cl_model.eval()

CLModule(
  (encoder_common): GeneExpressionBackbone(
    (act): ReLU()
    (encoder): GeneExpressionEncoder(
      (act1): ReLU()
      (act2): ReLU()
      (dropout): Dropout(p=0.10070817185386628, inplace=False)
      (layer1): Linear(in_features=4500, out_features=395, bias=True)
      (bn1): BatchNorm1d(395, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
      (layer2): Linear(in_features=395, out_features=128, bias=True)
      (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
    )
    (head): Sequential(
      (0): Linear(in_features=128, out_features=128, bias=True)
      (1): BatchNorm1d(128, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.10070817185386628, inplace=False)
      (4): Linear(in_features=128, out_features=64, bias=True)
    )
  )
  (encoder_male): GeneExpressionBackbone(
    (act): ReLU()
    (encoder): GeneExpressionEncoder(
      (act1): ReLU()
  

---

## CLF Part

In [35]:
from sklearn.preprocessing import StandardScaler

In [36]:
sum_hidden_1 = torch.zeros((x.shape[0], 128))
sum_hidden_2 = torch.zeros((x.shape[0], 64))
sum_hidden_3 = torch.zeros((x.shape[0], 32))
sum_pred = torch.zeros((x.shape[0], 1))

In [37]:
sum_clf_w1 = torch.zeros((128, 256))
sum_clf_w2 = torch.zeros((64, 128))
sum_clf_w3 = torch.zeros((32, 64))
sum_clf_w4 = torch.zeros((1, 32))

In [38]:
avg_hidden_1, avg_hidden_2, avg_hidden_3, avg_pred = [], [], [], []
avg_clf_w1, avg_clf_w2, avg_clf_w3, avg_clf_w4 = [], [], [], []
avg_embeddings = torch.zeros((x.shape[0], 256))
for experiment in range(1, 11):
    print("######################## %d experiment ########################" % (experiment))
    cl_net_hparams = [get_num_nodes(asthma), [395, 128, 64], 1, "he_normal", "relu", 0.10070817185386628] ### GSE240567
    cl_model = CLModule(cl_net_hparams, sparse_indices)
    cl_model.load_state_dict(torch.load(f"/home/koe3/Bioinformatics/Proposed/SHCL_v3/{asthma}_Result/{model_ver}/Saved_Model/[{model_date}_{model_num}]_[{experiment}]Opt_Model_StateDict.pth", map_location='cpu'))
    
    cl_model.eval()
    with torch.no_grad():
        cmn_embeddings, spf_embeddings = cl_model(x)
        embeddings = torch.cat((cmn_embeddings[0], spf_embeddings[0]), dim=1).cpu().detach().numpy()

        scaler = StandardScaler()
        scaled_embeddings = scaler.fit_transform(embeddings)
        x_embeddings = torch.from_numpy(scaled_embeddings).float()
        
    avg_embeddings += x_embeddings

    clf_net_hparams = [256, [128, 64, 32], 1, "he_normal", "sigmoid", 0.0006952330724237475]
    clf = Classifier(clf_net_hparams)
    clf.load_state_dict(torch.load(f"/home/koe3/Bioinformatics/Proposed/SHCL_v3/Opt_Params/SHCL/{asthma}/[{experiment}]Opt_Model_StateDict.pth", map_location='cpu'))
    clf.eval()
    with torch.no_grad():
        hidden_1 = clf.act1(clf.clf_bn1(clf.clf_layer1(x_embeddings)))
        hidden_2 = clf.act2(clf.clf_bn2(clf.clf_layer2(hidden_1)))
        hidden_3 = clf.act3(clf.clf_bn3(clf.clf_layer3(hidden_2)))
        pred = clf.output_activation(clf.clf_layer4(hidden_3))
        
        sum_hidden_1 += hidden_1
        sum_hidden_2 += hidden_2
        sum_hidden_3 += hidden_3
        sum_pred += pred
        
        sum_clf_w1 += clf.clf_layer1.weight.data
        sum_clf_w2 += clf.clf_layer2.weight.data
        sum_clf_w3 += clf.clf_layer3.weight.data
        sum_clf_w4 += clf.clf_layer4.weight.data

avg_embeddings /= 10

avg_hidden_1 = sum_hidden_1 / 10
avg_hidden_2 = sum_hidden_2 / 10
avg_hidden_3 = sum_hidden_3 / 10
avg_pred = sum_pred / 10

avg_clf_w1 = sum_clf_w1 / 10
avg_clf_w2 = sum_clf_w2 / 10
avg_clf_w3 = sum_clf_w3 / 10
avg_clf_w4 = sum_clf_w4 / 10

######################## 1 experiment ########################
######################## 2 experiment ########################
######################## 3 experiment ########################
######################## 4 experiment ########################
######################## 5 experiment ########################
######################## 6 experiment ########################
######################## 7 experiment ########################
######################## 8 experiment ########################
######################## 9 experiment ########################
######################## 10 experiment ########################


In [39]:
clf

Classifier(
  (clf_layer1): Linear(in_features=256, out_features=128, bias=True)
  (clf_bn1): BatchNorm1d(128, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
  (clf_layer2): Linear(in_features=128, out_features=64, bias=True)
  (clf_bn2): BatchNorm1d(64, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
  (clf_layer3): Linear(in_features=64, out_features=32, bias=True)
  (clf_bn3): BatchNorm1d(32, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
  (clf_layer4): Linear(in_features=32, out_features=1, bias=True)
  (act1): Sigmoid()
  (act2): Sigmoid()
  (act3): Sigmoid()
  (output_activation): Sigmoid()
  (dropout): Dropout(p=0.0006952330724237475, inplace=False)
)

In [40]:
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Hidden1_Nodes.csv", avg_hidden_1.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Hidden2_Nodes.csv", avg_hidden_2.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Hidden3_Nodes.csv", avg_hidden_3.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Pred.csv", avg_pred.cpu().detach().numpy(), delimiter = ",")

In [41]:
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Classifier_Layer1_Weights.csv", avg_clf_w1.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Classifier_Layer2_Weights.csv", avg_clf_w2.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Classifier_Layer3_Weights.csv", avg_clf_w3.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]{asthma}_AVG_Classifier_Layer4_Weights.csv", avg_clf_w4.cpu().detach().numpy(), delimiter = ",")

---

# Partial Derivatives

---

## Derivative of activation

In [52]:
def dSigmoid(x):
    sigmoid = 1 / (1 + np.exp(-x))
    return sigmoid * (1 - sigmoid)

In [53]:
def dReLU(x):
    if x <= 0:
        return 0
    elif x > 0:
        return 1

In [76]:
def dReLU(x):
    return (x > 0).astype(int)

---

In [46]:
avg_hidden_1 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Hidden1_Nodes.csv", header=None).values
avg_hidden_2 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Hidden2_Nodes.csv", header=None).values
avg_hidden_3 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Hidden3_Nodes.csv", header=None).values

In [47]:
avg_hidden_1.shape, avg_hidden_2.shape, avg_hidden_3.shape

((536, 128), (536, 64), (536, 32))

In [48]:
embeddings = avg_embeddings.cpu().detach().numpy()

In [49]:
embeddings, embeddings.shape

(array([[-5.5566561e-01, -6.1435550e-02, -1.0613473e-01, ...,
         -9.1741682e-04, -6.0762978e-01,  3.1936221e-02],
        [-1.0631281e-01, -1.9873326e-01,  2.7139026e-01, ...,
          1.6047570e-01, -1.5089318e-01,  1.2828538e-01],
        [ 3.0144583e-02,  1.1691929e+00,  1.4357926e+00, ...,
          1.9212270e-01, -5.0576741e-01,  1.5862366e+00],
        ...,
        [ 4.7558123e-01, -2.2080901e-01, -3.4206396e-01, ...,
          2.4712444e-03,  3.1666213e-01, -3.5117120e-01],
        [ 5.6916791e-01,  5.6515075e-02,  6.3895985e-02, ...,
         -9.9879041e-02,  5.7451153e-01,  2.6953727e-02],
        [ 6.3589126e-01,  3.2359439e-01,  2.0260167e-01, ...,
          1.0918860e-01,  8.1643057e-01, -1.8808153e-01]], dtype=float32),
 (536, 256))

In [50]:
avg_clf_w1 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Classifier_Layer1_Weights.csv", header=None).values
avg_clf_w2 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Classifier_Layer2_Weights.csv", header=None).values
avg_clf_w3 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Classifier_Layer3_Weights.csv", header=None).values
avg_clf_w4 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Classifier_Layer4_Weights.csv", header=None).values

In [51]:
avg_clf_w1.shape, avg_clf_w2.shape, avg_clf_w3.shape, avg_clf_w4.shape

((128, 256), (64, 128), (32, 64), (1, 32))

---

---

In [54]:
clf_hidden2_into_w3 = np.dot(avg_hidden_2, avg_clf_w3.T)
clf_hidden2_into_w3, clf_hidden2_into_w3.shape

(array([[ 0.41411308,  0.01814185,  0.13516911, ..., -0.07282797,
         -0.24419731, -0.25194135],
        [ 0.3470494 ,  0.03950484,  0.12229742, ..., -0.09182292,
         -0.26355092, -0.23754889],
        [ 0.26142085,  0.02053967,  0.0530617 , ..., -0.16589491,
         -0.28017902, -0.19813856],
        ...,
        [ 0.40974945,  0.02984776,  0.12454564, ..., -0.12887814,
         -0.21819466, -0.23452335],
        [ 0.35836946, -0.00768385,  0.03289345, ..., -0.13448937,
         -0.18208802, -0.20073954],
        [ 0.37617408, -0.02774087,  0.06869575, ..., -0.13422345,
         -0.16405729, -0.22106118]]),
 (536, 32))

In [55]:
df_hidden2_into_w3 = dSigmoid(clf_hidden2_into_w3)
df_hidden2_into_w3, df_hidden2_into_w3.shape

(array([[0.23958096, 0.24997943, 0.24886155, ..., 0.2496688 , 0.24630971,
         0.24607444],
        [0.24262087, 0.24990249, 0.24906753, ..., 0.24947377, 0.24570857,
         0.24650606],
        [0.24577688, 0.24997363, 0.24982411, ..., 0.24828779, 0.24515722,
         0.24756229],
        ...,
        [0.23979338, 0.24994433, 0.24903302, ..., 0.24896477, 0.2470479 ,
         0.24659369],
        [0.24214195, 0.24999631, 0.24993239, ..., 0.24887294, 0.24793914,
         0.2474983 ],
        [0.2413603 , 0.24995191, 0.24970529, ..., 0.24887738, 0.24832534,
         0.24697045]]),
 (536, 32))

In [56]:
clf_w4_into_df_hidden2_into_w3 = np.multiply(avg_clf_w4, df_hidden2_into_w3)
clf_w4_into_df_hidden2_into_w3, clf_w4_into_df_hidden2_into_w3.shape

(array([[ 0.01901082,  0.00532603,  0.0438299 , ...,  0.02955812,
         -0.03364687,  0.01954473],
        [ 0.01925204,  0.00532439,  0.04386618, ...,  0.02953503,
         -0.03356475,  0.01957901],
        [ 0.01950247,  0.00532591,  0.04399943, ...,  0.02939462,
         -0.03348943,  0.0196629 ],
        ...,
        [ 0.01902768,  0.00532528,  0.0438601 , ...,  0.02947477,
         -0.0337477 ,  0.01958597],
        [ 0.01921403,  0.00532639,  0.0440185 , ...,  0.02946389,
         -0.03386945,  0.01965782],
        [ 0.01915201,  0.00532544,  0.0439785 , ...,  0.02946442,
         -0.03392221,  0.01961589]]),
 (536, 32))

In [57]:
h2_updated = np.dot(clf_w4_into_df_hidden2_into_w3, avg_clf_w3)
h2_updated, h2_updated.shape

(array([[ 1.57312699e-03, -5.06885576e-04, -7.33539767e-05, ...,
         -6.04584990e-04,  5.62352339e-03, -5.29000790e-03],
        [ 1.55598805e-03, -5.35322149e-04, -8.37528485e-05, ...,
         -5.92641058e-04,  5.68159019e-03, -5.29500393e-03],
        [ 1.59348892e-03, -5.49869764e-04, -4.59071552e-05, ...,
         -5.75881395e-04,  5.71655283e-03, -5.30977694e-03],
        ...,
        [ 1.57296689e-03, -5.00258792e-04, -7.43661365e-05, ...,
         -6.24517190e-04,  5.62636465e-03, -5.29058849e-03],
        [ 1.58348852e-03, -5.04742399e-04, -5.59964534e-05, ...,
         -6.26048488e-04,  5.63955248e-03, -5.30184284e-03],
        [ 1.57546023e-03, -5.14016296e-04, -7.24621684e-05, ...,
         -6.40734993e-04,  5.62666736e-03, -5.31383659e-03]]),
 (536, 64))

---

In [58]:
clf_hidden1_into_w2 = np.dot(avg_hidden_1, avg_clf_w2.T)
clf_hidden1_into_w2, clf_hidden1_into_w2.shape

(array([[-0.08359708, -0.29130057, -0.19746023, ..., -0.02263397,
          0.07495492,  0.08536225],
        [-0.09022824, -0.31784196, -0.21115344, ...,  0.01694741,
          0.05456424,  0.10853905],
        [-0.09432922, -0.31917906, -0.14738937, ...,  0.05284354,
          0.0229947 ,  0.10699038],
        ...,
        [-0.05275015, -0.30108731, -0.30952756, ...,  0.00525026,
          0.05340761,  0.11486902],
        [-0.01634731, -0.31146789, -0.35297833, ...,  0.05388414,
          0.08458872,  0.16505654],
        [-0.00397814, -0.33035451, -0.34785242, ...,  0.06279468,
          0.07093112,  0.18906725]]),
 (536, 64))

In [59]:
df_hidden1_into_w2 = dSigmoid(clf_hidden1_into_w2)
df_hidden1_into_w2, df_hidden1_into_w2.shape

(array([[0.24956373, 0.24477061, 0.24757884, ..., 0.24996798, 0.24964919,
         0.24954513],
        [0.24949187, 0.24379084, 0.24723397, ..., 0.24998205, 0.24981401,
         0.24926515],
        [0.2494447 , 0.24373937, 0.24864717, ..., 0.24982555, 0.24996696,
         0.24928593],
        ...,
        [0.24982617, 0.24441867, 0.24410638, ..., 0.24999828, 0.24982181,
         0.24917713],
        [0.2499833 , 0.24403344, 0.24237179, ..., 0.24981862, 0.24955333,
         0.24830497],
        [0.24999901, 0.24330129, 0.24258736, ..., 0.24975371, 0.24968581,
         0.24777909]]),
 (536, 64))

In [60]:
h2_updated_into_df_hidden1_into_w2 = np.multiply(h2_updated, df_hidden1_into_w2)
h2_updated_into_df_hidden1_into_w2, h2_updated_into_df_hidden1_into_w2.shape

(array([[ 3.92595438e-04, -1.24070693e-04, -1.81608925e-05, ...,
         -1.51126891e-04,  1.40390805e-03, -1.32009573e-03],
        [ 3.88206366e-04, -1.30506636e-04, -2.07065489e-05, ...,
         -1.48149627e-04,  1.41934085e-03, -1.31985994e-03],
        [ 3.97487364e-04, -1.34024908e-04, -1.14146844e-05, ...,
         -1.43869888e-04,  1.42894931e-03, -1.32365268e-03],
        ...,
        [ 3.92968293e-04, -1.22272589e-04, -1.81532481e-05, ...,
         -1.56128221e-04,  1.40558861e-03, -1.31829365e-03],
        [ 3.95845682e-04, -1.23174024e-04, -1.35719606e-05, ...,
         -1.56398569e-04,  1.40736910e-03, -1.31647394e-03],
        [ 3.93863500e-04, -1.25060829e-04, -1.75784060e-05, ...,
         -1.60025944e-04,  1.40489901e-03, -1.31665760e-03]]),
 (536, 64))

In [61]:
h1_updated = np.dot(h2_updated_into_df_hidden1_into_w2, avg_clf_w2)
h1_updated, h1_updated.shape

(array([[ 0.00027953,  0.00067815, -0.00031129, ...,  0.00055955,
          0.00057342,  0.00029757],
        [ 0.00028164,  0.00067566, -0.00031077, ...,  0.00055972,
          0.00057565,  0.00029851],
        [ 0.00028279,  0.00067514, -0.00030666, ...,  0.00056422,
          0.00057195,  0.00029657],
        ...,
        [ 0.00028237,  0.00067954, -0.00031454, ...,  0.00055916,
          0.00057522,  0.00029989],
        [ 0.00028405,  0.00067753, -0.00031415, ...,  0.00055933,
          0.00057456,  0.00030069],
        [ 0.00028299,  0.000676  , -0.00031379, ...,  0.00055804,
          0.00057338,  0.00030186]]),
 (536, 128))

---

In [62]:
clf_e_into_w1 = np.dot(embeddings, avg_clf_w1.T)
clf_e_into_w1, clf_e_into_w1.shape

(array([[ 0.06567557,  0.07841577,  0.12592181, ..., -0.07996273,
          0.11911511,  0.01842429],
        [ 0.13620426,  0.03310102,  0.13286006, ..., -0.04331099,
          0.02131211,  0.0610714 ],
        [-0.30213976,  0.03502205,  0.14623195, ...,  0.36286232,
         -0.18549531,  0.10252722],
        ...,
        [-0.09856901,  0.13567279, -0.23719521, ..., -0.1418971 ,
          0.10865063,  0.00977145],
        [-0.0985239 , -0.10959432, -0.16133473, ..., -0.15950965,
          0.0605304 , -0.32460243],
        [-0.2624083 ,  0.0702165 , -0.21319676, ..., -0.16922557,
          0.09023475, -0.22733688]]),
 (536, 128))

In [63]:
df_e_into_w1 = dSigmoid(clf_e_into_w1)
df_e_into_w1, df_e_into_w1.shape

(array([[0.24973061, 0.24961608, 0.24901159, ..., 0.2496008 , 0.24911532,
         0.24997879],
        [0.2488441 , 0.24993153, 0.2489    , ..., 0.2498828 , 0.24997161,
         0.24976704],
        [0.24438017, 0.24992336, 0.24866826, ..., 0.24194796, 0.24786174,
         0.24934416],
        ...,
        [0.24939374, 0.24885308, 0.24651636, ..., 0.24874579, 0.24926364,
         0.24999403],
        [0.2493943 , 0.24925082, 0.24838023, ..., 0.24841651, 0.24977114,
         0.24352852],
        [0.24574528, 0.24969211, 0.24718058, ..., 0.24821868, 0.2494918 ,
         0.24679749]]),
 (536, 128))

In [64]:
h1_updated_into_df_e_into_w1 = np.multiply(h1_updated, df_e_into_w1)
h1_updated_into_df_e_into_w1, h1_updated_into_df_e_into_w1.shape

(array([[ 6.98077386e-05,  1.69278192e-04, -7.75141356e-05, ...,
          1.39663216e-04,  1.42847150e-04,  7.43855203e-05],
        [ 7.00837475e-05,  1.68868502e-04, -7.73506395e-05, ...,
          1.39864822e-04,  1.43897261e-04,  7.45589292e-05],
        [ 6.91071115e-05,  1.68732245e-04, -7.62560511e-05, ...,
          1.36512794e-04,  1.41764529e-04,  7.39484401e-05],
        ...,
        [ 7.04201630e-05,  1.69106803e-04, -7.75390410e-05, ...,
          1.39088536e-04,  1.43380968e-04,  7.49701686e-05],
        [ 7.08403646e-05,  1.68875784e-04, -7.80278515e-05, ...,
          1.38947611e-04,  1.43508498e-04,  7.32261211e-05],
        [ 6.95436246e-05,  1.68790867e-04, -7.75632441e-05, ...,
          1.38515729e-04,  1.43054808e-04,  7.44973163e-05]]),
 (536, 128))

In [65]:
e_updated = np.dot(h1_updated_into_df_e_into_w1, avg_clf_w1)
e_updated, e_updated.shape

(array([[-2.11411614e-05,  3.43906179e-05,  2.15982184e-05, ...,
          1.75018981e-05,  1.18566757e-05,  8.08115579e-06],
        [-2.11098159e-05,  3.43166127e-05,  2.15946492e-05, ...,
          1.75891637e-05,  1.18204611e-05,  8.19306219e-06],
        [-2.11140722e-05,  3.25253218e-05,  2.07047373e-05, ...,
          1.63184239e-05,  1.22723832e-05,  9.52775118e-06],
        ...,
        [-2.10858013e-05,  3.43951488e-05,  2.18384654e-05, ...,
          1.77381835e-05,  1.23037932e-05,  7.81368346e-06],
        [-2.11996270e-05,  3.40227428e-05,  2.17390000e-05, ...,
          1.78538956e-05,  1.15956203e-05,  8.01335380e-06],
        [-2.13278008e-05,  3.44618312e-05,  2.19845484e-05, ...,
          1.76966340e-05,  1.12650832e-05,  7.77278318e-06]]),
 (536, 256))

---

---

In [66]:
common_embeddings_updated = e_updated[:, :128]
male_spf_embeddings_updated = e_updated[male_indices, 128:]
female_spf_embeddings_updated = e_updated[female_indices, 128:]

In [67]:
common_embeddings_updated.shape, male_spf_embeddings_updated.shape, female_spf_embeddings_updated.shape

((536, 128), (206, 128), (330, 128))

In [68]:
avg_cmn_pathwy_node = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Common_Pathway_Nodes.csv", header=None).values
avg_cmn_repr_node = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Common_Representation_Nodes.csv", header=None).values
avg_cmn_hid_node = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Common_Head_Hidden_Nodes.csv", header=None).values
avg_cmn_proj_node = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Common_Projection_Nodes.csv", header=None).values

In [69]:
avg_male_pathwy_node = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Male_Pathway_Nodes.csv", header=None).values
avg_male_repr_node = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Male_Representation_Nodes.csv", header=None).values
avg_male_hid_node = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Male_Head_Hidden_Nodes.csv", header=None).values
avg_male_proj_node = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Male_Projection_Nodes.csv", header=None).values

In [70]:
avg_female_pathwy_node = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Female_Pathway_Nodes.csv", header=None).values
avg_female_repr_node = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Female_Representation_Nodes.csv", header=None).values
avg_female_hid_node = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Female_Head_Hidden_Nodes.csv", header=None).values
avg_female_proj_node = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Female_Projection_Nodes.csv", header=None).values

In [71]:
avg_cmn_w1 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Common_Encoder_Layer1_Weights.csv", header=None).values
avg_cmn_w2 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Common_Encoder_Layer2_Weights.csv", header=None).values
avg_cmn_w3 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Common_Head_Layer1_Weights.csv", header=None).values
avg_cmn_w4 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Common_Head_Layer2_Weights.csv", header=None).values

In [72]:
avg_male_w1 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Male_Encoder_Layer1_Weights.csv", header=None).values
avg_male_w2 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Male_Encoder_Layer2_Weights.csv", header=None).values
avg_male_w3 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Male_Head_Layer1_Weights.csv", header=None).values
avg_male_w4 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Male_Head_Layer2_Weights.csv", header=None).values

In [73]:
avg_female_w1 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Female_Encoder_Layer1_Weights.csv", header=None).values
avg_female_w2 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Female_Encoder_Layer2_Weights.csv", header=None).values
avg_female_w3 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Female_Head_Layer1_Weights.csv", header=None).values
avg_female_w4 = pd.read_csv(f"Opt_Params/[0911_1]GSE240567_AVG_Female_Head_Layer2_Weights.csv", header=None).values

---

## Sex-Common

In [74]:
cmn_pathway_into_w2 = np.dot(avg_cmn_pathwy_node, avg_cmn_w2.T)
cmn_pathway_into_w2, cmn_pathway_into_w2.shape

(array([[-0.28913311, -0.25249685, -0.15332945, ..., -0.24517948,
         -0.32655332, -0.10126908],
        [-0.30996519, -0.26959008, -0.13897586, ..., -0.38874235,
         -0.61235827, -0.12726378],
        [-1.54649598, -1.12052385, -0.60499381, ..., -1.35329003,
         -2.33968524, -0.49395377],
        ...,
        [-0.35376599, -0.46317411, -0.2876542 , ..., -0.38670426,
         -0.42206705, -0.32903579],
        [-0.25367165, -0.32283062, -0.21875891, ..., -0.3795586 ,
         -0.37965548, -0.29837142],
        [-0.36623593, -0.3153855 , -0.27856562, ..., -0.53330779,
         -0.33166278, -0.36522353]]),
 (536, 128))

In [77]:
df_cmn_pathway_into_w2 = dReLU(cmn_pathway_into_w2)
df_cmn_pathway_into_w2, df_cmn_pathway_into_w2.shape

(array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]]),
 (536, 128))

In [78]:
e_updated_into_df_cmn_pathway_into_w2 = np.multiply(common_embeddings_updated, df_cmn_pathway_into_w2)
e_updated_into_df_cmn_pathway_into_w2, e_updated_into_df_cmn_pathway_into_w2.shape

(array([[-0.,  0.,  0., ..., -0., -0., -0.],
        [-0.,  0.,  0., ..., -0., -0., -0.],
        [-0.,  0.,  0., ..., -0., -0., -0.],
        ...,
        [-0.,  0.,  0., ..., -0., -0., -0.],
        [-0.,  0.,  0., ..., -0., -0., -0.],
        [-0.,  0.,  0., ..., -0., -0., -0.]]),
 (536, 128))

In [79]:
cmn_pathway_updated = np.dot(e_updated_into_df_cmn_pathway_into_w2, avg_cmn_w2)
cmn_pathway_updated, cmn_pathway_updated.shape

(array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [-5.61454811e-07,  1.57898956e-07, -5.94548952e-07, ...,
          3.31966652e-07,  5.37740709e-07, -6.26975468e-08],
        ...,
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00]]),
 (536, 395))

---

In [80]:
cmn_pathway_df = pd.DataFrame(cmn_pathway_updated, columns=pathway_list)
cmn_pathway_df

,KEGG_ABC_TRANSPORTERS,KEGG_ACUTE_MYELOID_LEUKEMIA,KEGG_ADHERENS_JUNCTION,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,KEGG_ALDOSTERONE_REGULATED_SODIUM_REABSORPTION,KEGG_ALLOGRAFT_REJECTION,KEGG_ALPHA_LINOLENIC_ACID_METABOLISM,KEGG_ALZHEIMERS_DISEASE,KEGG_AMINOACYL_TRNA_BIOSYNTHESIS,...,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PRKAR1A_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH1_TO_HEDGEHOG_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_RASD1_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_UBQLN2_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_VCP_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_NOTCH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_MGLUR5_CA2_APOPTOTIC_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_PRNP_PI3K_NOX2_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_TRANSPORT_OF_CALCIUM
0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2,-5.614548e-07,1.578990e-07,-5.945490e-07,7.406862e-07,8.060121e-07,-7.609934e-07,-5.937606e-07,1.404273e-06,-8.118239e-07,7.958490e-07,...,-8.838204e-07,8.141121e-08,-6.640577e-07,1.346179e-07,2.816526e-07,7.001739e-07,-2.463758e-07,3.319667e-07,5.377407e-07,-6.269755e-08
3,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
4,1.389306e-06,-1.852863e-06,-3.123276e-06,-1.611477e-07,1.086189e-06,-9.631203e-07,-8.456293e-07,-6.112016e-08,-2.201620e-06,-5.117836e-07,...,1.485161e-06,1.884688e-07,-6.238637e-07,-2.456993e-06,-1.156831e-07,7.873992e-07,3.540841e-07,-2.449005e-07,1.679041e-06,-1.610822e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
532,6.341071e-08,-1.565463e-06,-2.229498e-07,-2.055774e-08,1.440887e-06,-4.092806e-08,2.604600e-07,-3.188025e-07,2.430571e-07,-2.284029e-07,...,5.563243e-07,-2.759723e-07,2.212620e-07,-5.596988e-07,-5.761613e-08,-9.671533e-07,8.213831e-07,7.778351e-07,-8.355369e-07,-6.552023e-07
533,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
534,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [81]:
cmn_pathway_df.to_csv(f"[{date}_{num}]{asthma}_Common_Pathway_Importance_Dataframe.csv", header = True, index = False)

---

In [82]:
gene_nodes = x[:, :-1].cpu().detach().numpy()
gene_nodes, gene_nodes.shape

(array([[ 0.4137391 ,  1.117053  ,  0.66359496, ..., -0.5035678 ,
         -1.4920847 , -0.8734951 ],
        [ 0.11444551,  2.1706772 ,  0.46172354, ...,  0.258784  ,
         -0.42835662, -1.5767702 ],
        [ 1.4437804 ,  1.1299856 , -0.5356963 , ..., -0.95408034,
          1.7762781 ,  1.8770624 ],
        ...,
        [ 1.1854957 ,  0.21329515,  0.9286507 , ..., -0.19325183,
         -0.46728823,  0.2538737 ],
        [ 0.23012996,  0.5196223 ,  0.3417612 , ..., -0.8669648 ,
         -1.1121025 , -0.9766483 ],
        [-0.72688115, -0.10760838,  1.001147  , ..., -1.6021441 ,
         -0.4519638 , -1.3123486 ]], dtype=float32),
 (536, 4500))

In [83]:
cmn_gene_into_w1 = np.dot(gene_nodes, avg_cmn_w1.T)
cmn_gene_into_w1, cmn_gene_into_w1.shape

(array([[ 0.03813446, -0.07156909, -0.0164821 , ..., -0.01977849,
         -0.00861334, -0.01769678],
        [-0.01714077, -0.09404853, -0.01864213, ..., -0.0620563 ,
         -0.04548433, -0.00645593],
        [-0.03233247,  0.03809187,  0.0407166 , ...,  0.03360104,
         -0.02916544,  0.06573702],
        ...,
        [ 0.01092911, -0.04488485,  0.02920071, ..., -0.00393262,
         -0.01591846, -0.00196652],
        [-0.02346806, -0.04207232, -0.02414335, ..., -0.051386  ,
         -0.02010493, -0.01168692],
        [-0.02112949, -0.01805189,  0.00060011, ..., -0.03332552,
          0.00799843, -0.04050985]]),
 (536, 395))

In [84]:
df_cmn_gene_into_w1 = dReLU(cmn_gene_into_w1)
df_cmn_gene_into_w1, df_cmn_gene_into_w1.shape

(array([[1, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 1, 1, ..., 1, 0, 1],
        ...,
        [1, 0, 1, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 1, ..., 0, 1, 0]]),
 (536, 395))

In [85]:
p_updated_into_df_cmn_gene_into_w1 = np.multiply(cmn_pathway_updated, df_cmn_gene_into_w1)
p_updated_into_df_cmn_gene_into_w1, p_updated_into_df_cmn_gene_into_w1.shape

(array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [-0.00000000e+00,  1.57898956e-07, -5.94548952e-07, ...,
          3.31966652e-07,  0.00000000e+00, -6.26975468e-08],
        ...,
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00]]),
 (536, 395))

In [86]:
cmn_gene_updated = np.dot(p_updated_into_df_cmn_gene_into_w1, avg_cmn_w1)
cmn_gene_updated, cmn_gene_updated.shape

(array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  1.29757659e-08,  1.87128361e-09, ...,
          8.45634641e-11, -1.78471890e-08,  1.58207097e-08],
        ...,
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00]]),
 (536, 4500))

In [87]:
data.columns[:-1]

Index(['DPM1', 'FGR', 'CFH', 'FUCA2', 'GCLC', 'NFYA', 'SEMA3F', 'CFTR',
       'CYP51A1', 'RAD52',
       ...
       'H2AC17', 'H4C2', 'H3C10', 'ADORA3', 'PIGY', 'H3C2', 'H3C3', 'NPBWR1',
       'UGT1A3', 'UGT1A5'],
      dtype='object', length=4500)

In [88]:
cmn_gene_df = pd.DataFrame(cmn_gene_updated, columns=data.columns[:-1])
cmn_gene_df

,DPM1,FGR,CFH,FUCA2,GCLC,NFYA,SEMA3F,CFTR,CYP51A1,RAD52,...,H2AC17,H4C2,H3C10,ADORA3,PIGY,H3C2,H3C3,NPBWR1,UGT1A3,UGT1A5
0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2,0.000000e+00,1.297577e-08,1.871284e-09,3.843207e-10,0.0,0.000000e+00,-1.586767e-09,1.242851e-10,0.000000e+00,-5.739490e-10,...,2.729920e-09,1.672082e-09,4.708784e-10,1.042283e-10,0.000000e+00,-3.993251e-09,3.887306e-09,8.456346e-11,-1.784719e-08,1.582071e-08
3,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
4,8.927430e-09,0.000000e+00,0.000000e+00,1.346626e-10,0.0,-1.091458e-08,0.000000e+00,0.000000e+00,6.107713e-09,-8.839507e-09,...,0.000000e+00,0.000000e+00,-3.788405e-09,0.000000e+00,-2.890757e-09,6.257296e-09,-1.310120e-09,0.000000e+00,2.575537e-09,-4.449813e-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
532,0.000000e+00,0.000000e+00,0.000000e+00,-1.357786e-09,0.0,-4.472802e-09,0.000000e+00,1.017437e-11,0.000000e+00,0.000000e+00,...,3.434133e-09,2.103414e-09,-2.073065e-09,-7.413477e-09,0.000000e+00,-3.421358e-09,3.968318e-09,-6.014770e-09,-8.019709e-10,-4.466618e-09
533,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
534,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [89]:
cmn_gene_df.to_csv(f"[{date}_{num}]{asthma}_Common_Gene_Importance_Dataframe.csv", header = True, index = False)

---

---

## Male-Specific

In [90]:
male_pathway_into_w2 = np.dot(avg_male_pathwy_node, avg_male_w2.T)
male_pathway_into_w2, male_pathway_into_w2.shape

(array([[-0.40565695, -0.19550762, -0.16312115, ..., -0.29387254,
         -0.42749847, -0.20770992],
        [-1.29098329, -0.76172024, -0.47386997, ..., -1.20489001,
         -1.3962843 , -0.65230055],
        [-0.60588432, -0.29143132, -0.28368336, ..., -0.42611095,
         -0.65776542, -0.20010076],
        ...,
        [-0.40832215, -0.1913095 , -0.18698432, ..., -0.42654319,
         -0.1916039 , -0.24937862],
        [-0.36257764, -0.25065974, -0.32336319, ..., -0.37286843,
         -0.2384174 , -0.29452358],
        [-0.3409345 , -0.2690611 , -0.23926165, ..., -0.46956654,
         -0.31847773, -0.29951444]]),
 (206, 128))

In [91]:
df_male_pathway_into_w2 = dReLU(male_pathway_into_w2)
df_male_pathway_into_w2, df_male_pathway_into_w2.shape

(array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]]),
 (206, 128))

In [92]:
e_updated_into_df_male_pathway_into_w2 = np.multiply(male_spf_embeddings_updated, df_male_pathway_into_w2)
e_updated_into_df_male_pathway_into_w2, e_updated_into_df_male_pathway_into_w2.shape

(array([[ 0.,  0., -0., ...,  0.,  0.,  0.],
        [ 0.,  0., -0., ...,  0.,  0.,  0.],
        [ 0.,  0., -0., ...,  0.,  0.,  0.],
        ...,
        [ 0.,  0., -0., ...,  0.,  0.,  0.],
        [ 0.,  0., -0., ...,  0.,  0.,  0.],
        [ 0.,  0., -0., ...,  0.,  0.,  0.]]),
 (206, 128))

In [93]:
male_pathway_updated = np.dot(e_updated_into_df_male_pathway_into_w2, avg_male_w2)
male_pathway_updated, male_pathway_updated.shape

(array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 1.00736658e-06,  1.80198719e-06,  7.26211405e-07, ...,
         -4.23833376e-07,  4.05267202e-07, -5.78764810e-08],
        ...,
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00]]),
 (206, 395))

---

In [94]:
male_pathway_df = pd.DataFrame(male_pathway_updated, columns=pathway_list)
male_pathway_df

,KEGG_ABC_TRANSPORTERS,KEGG_ACUTE_MYELOID_LEUKEMIA,KEGG_ADHERENS_JUNCTION,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,KEGG_ALDOSTERONE_REGULATED_SODIUM_REABSORPTION,KEGG_ALLOGRAFT_REJECTION,KEGG_ALPHA_LINOLENIC_ACID_METABOLISM,KEGG_ALZHEIMERS_DISEASE,KEGG_AMINOACYL_TRNA_BIOSYNTHESIS,...,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PRKAR1A_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH1_TO_HEDGEHOG_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_RASD1_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_UBQLN2_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_VCP_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_NOTCH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_MGLUR5_CA2_APOPTOTIC_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_PRNP_PI3K_NOX2_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_TRANSPORT_OF_CALCIUM
0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2,1.007367e-06,1.801987e-06,7.262114e-07,-7.932771e-09,-8.129619e-07,-1.184159e-06,-5.444161e-07,8.915670e-07,8.198332e-07,-2.401952e-07,...,1.242610e-06,-5.787053e-07,8.930545e-07,-1.707027e-07,1.377533e-06,2.683743e-07,-9.447101e-07,-4.238334e-07,4.052672e-07,-5.787648e-08
3,-3.608778e-07,-6.399077e-07,2.398108e-07,3.033449e-07,-3.450770e-07,9.313481e-08,5.324113e-07,-3.930735e-07,1.334819e-07,-1.681436e-07,...,2.102258e-07,9.182274e-07,1.223869e-07,7.413295e-07,-1.358482e-07,2.445539e-07,-2.099023e-07,1.637926e-07,-1.800852e-07,-3.267663e-07
4,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
202,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
203,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
204,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [95]:
male_pathway_df.to_csv(f"[{date}_{num}]{asthma}_Male_Pathway_Importance_Dataframe.csv", header = True, index = False)

---

In [96]:
gene_nodes = x[:, :-1].cpu().detach().numpy()
gene_nodes, gene_nodes.shape

(array([[ 0.4137391 ,  1.117053  ,  0.66359496, ..., -0.5035678 ,
         -1.4920847 , -0.8734951 ],
        [ 0.11444551,  2.1706772 ,  0.46172354, ...,  0.258784  ,
         -0.42835662, -1.5767702 ],
        [ 1.4437804 ,  1.1299856 , -0.5356963 , ..., -0.95408034,
          1.7762781 ,  1.8770624 ],
        ...,
        [ 1.1854957 ,  0.21329515,  0.9286507 , ..., -0.19325183,
         -0.46728823,  0.2538737 ],
        [ 0.23012996,  0.5196223 ,  0.3417612 , ..., -0.8669648 ,
         -1.1121025 , -0.9766483 ],
        [-0.72688115, -0.10760838,  1.001147  , ..., -1.6021441 ,
         -0.4519638 , -1.3123486 ]], dtype=float32),
 (536, 4500))

In [97]:
gene_nodes[male_indices], gene_nodes[male_indices].shape

(array([[ 0.4137391 ,  1.117053  ,  0.66359496, ..., -0.5035678 ,
         -1.4920847 , -0.8734951 ],
        [ 1.4437804 ,  1.1299856 , -0.5356963 , ..., -0.95408034,
          1.7762781 ,  1.8770624 ],
        [ 0.3583226 ,  0.26114896,  0.6647537 , ..., -0.19360308,
          0.34701732,  0.1178697 ],
        ...,
        [-0.32322234,  0.14556624,  0.3610977 , ..., -0.70825046,
         -2.6766803 , -2.2380345 ],
        [-0.8456857 ,  0.49453494,  0.9441822 , ..., -1.8655412 ,
         -0.6131131 , -0.2645199 ],
        [ 1.1854957 ,  0.21329515,  0.9286507 , ..., -0.19325183,
         -0.46728823,  0.2538737 ]], dtype=float32),
 (206, 4500))

In [98]:
male_gene_into_w1 = np.dot(gene_nodes[male_indices], avg_male_w1.T)
male_gene_into_w1, male_gene_into_w1.shape

(array([[-0.05468506,  0.0673277 , -0.00671739, ..., -0.02340553,
          0.00275226,  0.01487205],
        [ 0.03725607,  0.01486473,  0.07839366, ..., -0.08361965,
         -0.01381238,  0.00438696],
        [ 0.03702219,  0.00733676,  0.00943849, ..., -0.0699189 ,
          0.00942305,  0.03025754],
        ...,
        [-0.04525198,  0.00485297, -0.05281545, ...,  0.02109346,
          0.00055388,  0.02695731],
        [ 0.0320774 , -0.01825606, -0.03896112, ...,  0.01734436,
          0.00443488,  0.01277998],
        [-0.03622496, -0.04221326, -0.03080534, ..., -0.0109979 ,
          0.00225157,  0.01484814]]),
 (206, 395))

In [99]:
df_male_gene_into_w1 = dReLU(male_gene_into_w1)
df_male_gene_into_w1, df_male_gene_into_w1.shape

(array([[0, 1, 0, ..., 0, 1, 1],
        [1, 1, 1, ..., 0, 0, 1],
        [1, 1, 1, ..., 0, 1, 1],
        ...,
        [0, 1, 0, ..., 1, 1, 1],
        [1, 0, 0, ..., 1, 1, 1],
        [0, 0, 0, ..., 0, 1, 1]]),
 (206, 395))

In [100]:
p_updated_into_df_male_gene_into_w1 = np.multiply(male_pathway_updated, df_male_gene_into_w1)
p_updated_into_df_male_gene_into_w1, p_updated_into_df_male_gene_into_w1.shape

(array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 1.00736658e-06,  1.80198719e-06,  7.26211405e-07, ...,
         -0.00000000e+00,  4.05267202e-07, -5.78764810e-08],
        ...,
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00]]),
 (206, 395))

In [101]:
male_gene_updated = np.dot(p_updated_into_df_male_gene_into_w1, avg_male_w1)
male_gene_updated, male_gene_updated.shape

(array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 1.35438704e-09, 0.00000000e+00, ...,
         0.00000000e+00, 6.66653848e-10, 2.54846441e-10],
        ...,
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00]]),
 (206, 4500))

In [102]:
male_gene_df = pd.DataFrame(male_gene_updated, columns=data.columns[:-1])
male_gene_df

,DPM1,FGR,CFH,FUCA2,GCLC,NFYA,SEMA3F,CFTR,CYP51A1,RAD52,...,H2AC17,H4C2,H3C10,ADORA3,PIGY,H3C2,H3C3,NPBWR1,UGT1A3,UGT1A5
0,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
1,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
2,0.0,1.354387e-09,0.0,5.337665e-09,3.921949e-10,0.000000e+00,0.0,-2.149358e-09,-1.475610e-09,1.331720e-08,...,4.402699e-10,2.004573e-09,4.758564e-09,0.0,1.574215e-08,1.826950e-09,-1.096146e-09,0.0,6.666538e-10,2.548464e-10
3,0.0,0.000000e+00,0.0,-2.004153e-09,0.000000e+00,6.750605e-12,0.0,7.699835e-10,-4.253534e-10,0.000000e+00,...,-5.511608e-10,-2.509466e-09,-5.755806e-09,0.0,-5.116951e-10,-8.648935e-10,2.184997e-09,0.0,-4.009709e-09,1.633940e-09
4,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
202,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
203,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
204,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00


In [103]:
male_gene_df.to_csv(f"[{date}_{num}]{asthma}_Male_Gene_Importance_Dataframe.csv", header = True, index = False)

---

---

## Female-Specific

In [104]:
female_pathway_into_w2 = np.dot(avg_female_pathwy_node, avg_female_w2.T)
female_pathway_into_w2, female_pathway_into_w2.shape

(array([[-0.25072926, -0.34452417, -0.27571719, ..., -0.33171393,
         -0.5737021 , -0.33219473],
        [-0.30029438, -0.47330498, -0.39722636, ..., -0.31238182,
         -0.54014651, -0.35016591],
        [-0.69009277, -0.87540392, -0.66069209, ..., -0.27509891,
         -1.02198078, -0.45026649],
        ...,
        [-0.33523103, -0.30368251, -0.28772683, ..., -0.34403796,
         -0.31415237, -0.41211015],
        [-0.25530903, -0.46354381, -0.43263758, ..., -0.34153187,
         -0.42539952, -0.45401849],
        [-0.1835114 , -0.35291523, -0.36097055, ..., -0.31332657,
         -0.43256769, -0.53099865]]),
 (330, 128))

In [105]:
df_female_pathway_into_w2 = dReLU(female_pathway_into_w2)
df_female_pathway_into_w2, df_female_pathway_into_w2.shape

(array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]]),
 (330, 128))

In [106]:
e_updated_into_df_female_pathway_into_w2 = np.multiply(female_spf_embeddings_updated, df_female_pathway_into_w2)
e_updated_into_df_female_pathway_into_w2, e_updated_into_df_female_pathway_into_w2.shape

(array([[ 0.,  0., -0., ...,  0.,  0.,  0.],
        [ 0.,  0., -0., ...,  0.,  0.,  0.],
        [ 0.,  0., -0., ...,  0.,  0.,  0.],
        ...,
        [ 0.,  0., -0., ...,  0.,  0.,  0.],
        [ 0.,  0., -0., ...,  0.,  0.,  0.],
        [ 0.,  0., -0., ...,  0.,  0.,  0.]]),
 (330, 128))

In [107]:
female_pathway_updated = np.dot(e_updated_into_df_female_pathway_into_w2, avg_female_w2)
female_pathway_updated, female_pathway_updated.shape

(array([[ 9.30559282e-08, -1.77952194e-07, -3.57910371e-07, ...,
         -2.54916676e-07,  9.50537180e-08,  1.32806991e-07],
        [-4.17008265e-08,  2.05534991e-07,  3.91584683e-07, ...,
         -6.68833670e-07,  8.50088195e-07,  2.18838085e-07],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        ...,
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00]]),
 (330, 395))

---

In [108]:
female_pathway_df = pd.DataFrame(female_pathway_updated, columns=pathway_list)
female_pathway_df

,KEGG_ABC_TRANSPORTERS,KEGG_ACUTE_MYELOID_LEUKEMIA,KEGG_ADHERENS_JUNCTION,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,KEGG_ALDOSTERONE_REGULATED_SODIUM_REABSORPTION,KEGG_ALLOGRAFT_REJECTION,KEGG_ALPHA_LINOLENIC_ACID_METABOLISM,KEGG_ALZHEIMERS_DISEASE,KEGG_AMINOACYL_TRNA_BIOSYNTHESIS,...,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PRKAR1A_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH1_TO_HEDGEHOG_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_RASD1_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_UBQLN2_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_VCP_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_NOTCH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_MGLUR5_CA2_APOPTOTIC_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_PRNP_PI3K_NOX2_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_TRANSPORT_OF_CALCIUM
0,9.305593e-08,-1.779522e-07,-3.579104e-07,6.521932e-08,9.887262e-08,3.569176e-07,9.884956e-08,8.703743e-08,2.856533e-07,-5.625486e-07,...,4.528555e-07,-1.827904e-07,1.105048e-07,1.925378e-07,-3.295670e-07,3.594232e-08,2.295983e-07,-2.549167e-07,9.505372e-08,1.328070e-07
1,-4.170083e-08,2.055350e-07,3.915847e-07,5.657998e-07,4.446641e-07,9.665850e-08,4.928081e-07,-1.312741e-06,6.490300e-07,2.823190e-09,...,1.307644e-07,4.494804e-07,1.859973e-07,-9.997643e-08,-2.480486e-07,2.567429e-07,4.677592e-07,-6.688337e-07,8.500882e-07,2.188381e-07
2,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
3,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
4,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
326,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
327,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
328,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [109]:
female_pathway_df.to_csv(f"[{date}_{num}]{asthma}_Female_Pathway_Importance_Dataframe.csv", header = True, index = False)

---

In [110]:
gene_nodes = x[:, :-1].cpu().detach().numpy()
gene_nodes, gene_nodes.shape

(array([[ 0.4137391 ,  1.117053  ,  0.66359496, ..., -0.5035678 ,
         -1.4920847 , -0.8734951 ],
        [ 0.11444551,  2.1706772 ,  0.46172354, ...,  0.258784  ,
         -0.42835662, -1.5767702 ],
        [ 1.4437804 ,  1.1299856 , -0.5356963 , ..., -0.95408034,
          1.7762781 ,  1.8770624 ],
        ...,
        [ 1.1854957 ,  0.21329515,  0.9286507 , ..., -0.19325183,
         -0.46728823,  0.2538737 ],
        [ 0.23012996,  0.5196223 ,  0.3417612 , ..., -0.8669648 ,
         -1.1121025 , -0.9766483 ],
        [-0.72688115, -0.10760838,  1.001147  , ..., -1.6021441 ,
         -0.4519638 , -1.3123486 ]], dtype=float32),
 (536, 4500))

In [111]:
gene_nodes[female_indices], gene_nodes[female_indices].shape

(array([[ 0.11444551,  2.1706772 ,  0.46172354, ...,  0.258784  ,
         -0.42835662, -1.5767702 ],
        [-1.089564  ,  0.07186296, -1.6474705 , ...,  1.1880821 ,
          0.5720939 ,  1.0506992 ],
        [ 1.3655012 ,  0.17964932,  0.51349264, ..., -0.09861358,
          0.37643307,  0.10667439],
        ...,
        [ 0.08479113, -0.307321  ,  0.44416326, ...,  0.5421178 ,
         -0.79749256, -0.9648889 ],
        [ 0.23012996,  0.5196223 ,  0.3417612 , ..., -0.8669648 ,
         -1.1121025 , -0.9766483 ],
        [-0.72688115, -0.10760838,  1.001147  , ..., -1.6021441 ,
         -0.4519638 , -1.3123486 ]], dtype=float32),
 (330, 4500))

In [112]:
female_gene_into_w1 = np.dot(gene_nodes[female_indices], avg_female_w1.T)
female_gene_into_w1, female_gene_into_w1.shape

(array([[-0.00026908, -0.0226483 ,  0.02004196, ...,  0.00622767,
          0.0136461 ,  0.01666152],
        [ 0.01962702, -0.01057967, -0.02848359, ...,  0.02126308,
          0.00731844,  0.00960779],
        [-0.01797183,  0.11405198, -0.02976367, ..., -0.06438285,
          0.01255692, -0.01167002],
        ...,
        [ 0.03945272,  0.00694465, -0.05204671, ...,  0.02113274,
          0.00175961,  0.0193095 ],
        [-0.01920764, -0.05294125, -0.02723155, ...,  0.00179299,
          0.01618206,  0.02954713],
        [ 0.01118887, -0.00431164,  0.01201263, ...,  0.0143109 ,
          0.02786206, -0.00243184]]),
 (330, 395))

In [113]:
df_female_gene_into_w1 = dReLU(female_gene_into_w1)
df_female_gene_into_w1, df_female_gene_into_w1.shape

(array([[0, 0, 1, ..., 1, 1, 1],
        [1, 0, 0, ..., 1, 1, 1],
        [0, 1, 0, ..., 0, 1, 0],
        ...,
        [1, 1, 0, ..., 1, 1, 1],
        [0, 0, 0, ..., 1, 1, 1],
        [1, 0, 1, ..., 1, 1, 0]]),
 (330, 395))

In [114]:
p_updated_into_df_female_gene_into_w1 = np.multiply(female_pathway_updated, df_female_gene_into_w1)
p_updated_into_df_female_gene_into_w1, p_updated_into_df_female_gene_into_w1.shape

(array([[ 0.00000000e+00, -0.00000000e+00, -3.57910371e-07, ...,
         -2.54916676e-07,  9.50537180e-08,  1.32806991e-07],
        [-4.17008265e-08,  0.00000000e+00,  0.00000000e+00, ...,
         -6.68833670e-07,  8.50088195e-07,  2.18838085e-07],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        ...,
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00]]),
 (330, 395))

In [115]:
female_gene_updated = np.dot(p_updated_into_df_female_gene_into_w1, avg_female_w1)
female_gene_updated, female_gene_updated.shape

(array([[ 3.45721178e-10,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00, -4.53964278e-09,  4.33657711e-09],
        [ 3.71994018e-09, -7.27430381e-10,  1.30798619e-09, ...,
          0.00000000e+00,  1.52937821e-10,  1.52553728e-08],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        ...,
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00]]),
 (330, 4500))

In [116]:
female_gene_df = pd.DataFrame(female_gene_updated, columns=data.columns[:-1])
female_gene_df

,DPM1,FGR,CFH,FUCA2,GCLC,NFYA,SEMA3F,CFTR,CYP51A1,RAD52,...,H2AC17,H4C2,H3C10,ADORA3,PIGY,H3C2,H3C3,NPBWR1,UGT1A3,UGT1A5
0,3.457212e-10,0.000000e+00,0.000000e+00,-1.813468e-10,-7.852618e-12,0.000000e+00,0.000000e+00,-1.804326e-10,0.0,-1.310343e-09,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,-1.897135e-10,0.000000e+00,0.000000e+00,0.0,-4.539643e-09,4.336577e-09
1,3.719940e-09,-7.274304e-10,1.307986e-09,0.000000e+00,-2.692568e-11,-9.495614e-10,2.300435e-09,-3.233778e-11,0.0,0.000000e+00,...,-2.624130e-10,2.856142e-10,-2.754221e-10,0.0,3.161044e-10,2.199296e-09,2.023722e-09,0.0,1.529378e-10,1.525537e-08
2,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
3,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
4,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
326,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
327,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
328,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00


In [117]:
female_gene_df.to_csv(f"[{date}_{num}]{asthma}_Female_Gene_Importance_Dataframe.csv", header = True, index = False)

---

---

In [118]:
cmn_pathway_importance_df = cmn_pathway_df.abs()
cmn_pathway_importance_df

,KEGG_ABC_TRANSPORTERS,KEGG_ACUTE_MYELOID_LEUKEMIA,KEGG_ADHERENS_JUNCTION,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,KEGG_ALDOSTERONE_REGULATED_SODIUM_REABSORPTION,KEGG_ALLOGRAFT_REJECTION,KEGG_ALPHA_LINOLENIC_ACID_METABOLISM,KEGG_ALZHEIMERS_DISEASE,KEGG_AMINOACYL_TRNA_BIOSYNTHESIS,...,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PRKAR1A_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH1_TO_HEDGEHOG_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_RASD1_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_UBQLN2_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_VCP_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_NOTCH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_MGLUR5_CA2_APOPTOTIC_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_PRNP_PI3K_NOX2_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_TRANSPORT_OF_CALCIUM
0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2,5.614548e-07,1.578990e-07,5.945490e-07,7.406862e-07,8.060121e-07,7.609934e-07,5.937606e-07,1.404273e-06,8.118239e-07,7.958490e-07,...,8.838204e-07,8.141121e-08,6.640577e-07,1.346179e-07,2.816526e-07,7.001739e-07,2.463758e-07,3.319667e-07,5.377407e-07,6.269755e-08
3,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
4,1.389306e-06,1.852863e-06,3.123276e-06,1.611477e-07,1.086189e-06,9.631203e-07,8.456293e-07,6.112016e-08,2.201620e-06,5.117836e-07,...,1.485161e-06,1.884688e-07,6.238637e-07,2.456993e-06,1.156831e-07,7.873992e-07,3.540841e-07,2.449005e-07,1.679041e-06,1.610822e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
532,6.341071e-08,1.565463e-06,2.229498e-07,2.055774e-08,1.440887e-06,4.092806e-08,2.604600e-07,3.188025e-07,2.430571e-07,2.284029e-07,...,5.563243e-07,2.759723e-07,2.212620e-07,5.596988e-07,5.761613e-08,9.671533e-07,8.213831e-07,7.778351e-07,8.355369e-07,6.552023e-07
533,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
534,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [119]:
cmn_feature_importance_df = cmn_gene_df.abs()
cmn_feature_importance_df

,DPM1,FGR,CFH,FUCA2,GCLC,NFYA,SEMA3F,CFTR,CYP51A1,RAD52,...,H2AC17,H4C2,H3C10,ADORA3,PIGY,H3C2,H3C3,NPBWR1,UGT1A3,UGT1A5
0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2,0.000000e+00,1.297577e-08,1.871284e-09,3.843207e-10,0.0,0.000000e+00,1.586767e-09,1.242851e-10,0.000000e+00,5.739490e-10,...,2.729920e-09,1.672082e-09,4.708784e-10,1.042283e-10,0.000000e+00,3.993251e-09,3.887306e-09,8.456346e-11,1.784719e-08,1.582071e-08
3,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
4,8.927430e-09,0.000000e+00,0.000000e+00,1.346626e-10,0.0,1.091458e-08,0.000000e+00,0.000000e+00,6.107713e-09,8.839507e-09,...,0.000000e+00,0.000000e+00,3.788405e-09,0.000000e+00,2.890757e-09,6.257296e-09,1.310120e-09,0.000000e+00,2.575537e-09,4.449813e-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
532,0.000000e+00,0.000000e+00,0.000000e+00,1.357786e-09,0.0,4.472802e-09,0.000000e+00,1.017437e-11,0.000000e+00,0.000000e+00,...,3.434133e-09,2.103414e-09,2.073065e-09,7.413477e-09,0.000000e+00,3.421358e-09,3.968318e-09,6.014770e-09,8.019709e-10,4.466618e-09
533,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
534,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [120]:
male_pathway_importance_df = male_pathway_df.abs()
male_pathway_importance_df

,KEGG_ABC_TRANSPORTERS,KEGG_ACUTE_MYELOID_LEUKEMIA,KEGG_ADHERENS_JUNCTION,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,KEGG_ALDOSTERONE_REGULATED_SODIUM_REABSORPTION,KEGG_ALLOGRAFT_REJECTION,KEGG_ALPHA_LINOLENIC_ACID_METABOLISM,KEGG_ALZHEIMERS_DISEASE,KEGG_AMINOACYL_TRNA_BIOSYNTHESIS,...,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PRKAR1A_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH1_TO_HEDGEHOG_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_RASD1_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_UBQLN2_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_VCP_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_NOTCH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_MGLUR5_CA2_APOPTOTIC_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_PRNP_PI3K_NOX2_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_TRANSPORT_OF_CALCIUM
0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2,1.007367e-06,1.801987e-06,7.262114e-07,7.932771e-09,8.129619e-07,1.184159e-06,5.444161e-07,8.915670e-07,8.198332e-07,2.401952e-07,...,1.242610e-06,5.787053e-07,8.930545e-07,1.707027e-07,1.377533e-06,2.683743e-07,9.447101e-07,4.238334e-07,4.052672e-07,5.787648e-08
3,3.608778e-07,6.399077e-07,2.398108e-07,3.033449e-07,3.450770e-07,9.313481e-08,5.324113e-07,3.930735e-07,1.334819e-07,1.681436e-07,...,2.102258e-07,9.182274e-07,1.223869e-07,7.413295e-07,1.358482e-07,2.445539e-07,2.099023e-07,1.637926e-07,1.800852e-07,3.267663e-07
4,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
202,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
203,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
204,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [121]:
male_feature_importance_df = male_gene_df.abs()
male_feature_importance_df

,DPM1,FGR,CFH,FUCA2,GCLC,NFYA,SEMA3F,CFTR,CYP51A1,RAD52,...,H2AC17,H4C2,H3C10,ADORA3,PIGY,H3C2,H3C3,NPBWR1,UGT1A3,UGT1A5
0,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
1,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
2,0.0,1.354387e-09,0.0,5.337665e-09,3.921949e-10,0.000000e+00,0.0,2.149358e-09,1.475610e-09,1.331720e-08,...,4.402699e-10,2.004573e-09,4.758564e-09,0.0,1.574215e-08,1.826950e-09,1.096146e-09,0.0,6.666538e-10,2.548464e-10
3,0.0,0.000000e+00,0.0,2.004153e-09,0.000000e+00,6.750605e-12,0.0,7.699835e-10,4.253534e-10,0.000000e+00,...,5.511608e-10,2.509466e-09,5.755806e-09,0.0,5.116951e-10,8.648935e-10,2.184997e-09,0.0,4.009709e-09,1.633940e-09
4,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
202,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
203,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
204,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00


In [122]:
female_pathway_importance_df = female_pathway_df.abs()
female_pathway_importance_df

,KEGG_ABC_TRANSPORTERS,KEGG_ACUTE_MYELOID_LEUKEMIA,KEGG_ADHERENS_JUNCTION,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,KEGG_ALDOSTERONE_REGULATED_SODIUM_REABSORPTION,KEGG_ALLOGRAFT_REJECTION,KEGG_ALPHA_LINOLENIC_ACID_METABOLISM,KEGG_ALZHEIMERS_DISEASE,KEGG_AMINOACYL_TRNA_BIOSYNTHESIS,...,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PRKAR1A_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH1_TO_HEDGEHOG_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_RASD1_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_UBQLN2_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_VCP_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_NOTCH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_MGLUR5_CA2_APOPTOTIC_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_PRNP_PI3K_NOX2_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_TRANSPORT_OF_CALCIUM
0,9.305593e-08,1.779522e-07,3.579104e-07,6.521932e-08,9.887262e-08,3.569176e-07,9.884956e-08,8.703743e-08,2.856533e-07,5.625486e-07,...,4.528555e-07,1.827904e-07,1.105048e-07,1.925378e-07,3.295670e-07,3.594232e-08,2.295983e-07,2.549167e-07,9.505372e-08,1.328070e-07
1,4.170083e-08,2.055350e-07,3.915847e-07,5.657998e-07,4.446641e-07,9.665850e-08,4.928081e-07,1.312741e-06,6.490300e-07,2.823190e-09,...,1.307644e-07,4.494804e-07,1.859973e-07,9.997643e-08,2.480486e-07,2.567429e-07,4.677592e-07,6.688337e-07,8.500882e-07,2.188381e-07
2,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
3,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
4,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
326,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
327,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
328,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [123]:
female_feature_importance_df = female_gene_df.abs()
female_feature_importance_df

,DPM1,FGR,CFH,FUCA2,GCLC,NFYA,SEMA3F,CFTR,CYP51A1,RAD52,...,H2AC17,H4C2,H3C10,ADORA3,PIGY,H3C2,H3C3,NPBWR1,UGT1A3,UGT1A5
0,3.457212e-10,0.000000e+00,0.000000e+00,1.813468e-10,7.852618e-12,0.000000e+00,0.000000e+00,1.804326e-10,0.0,1.310343e-09,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,1.897135e-10,0.000000e+00,0.000000e+00,0.0,4.539643e-09,4.336577e-09
1,3.719940e-09,7.274304e-10,1.307986e-09,0.000000e+00,2.692568e-11,9.495614e-10,2.300435e-09,3.233778e-11,0.0,0.000000e+00,...,2.624130e-10,2.856142e-10,2.754221e-10,0.0,3.161044e-10,2.199296e-09,2.023722e-09,0.0,1.529378e-10,1.525537e-08
2,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
3,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
4,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
326,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
327,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
328,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00


---

---

In [124]:
cmn_feature_importance_df.mean().sort_values(ascending=False)

PIK3R1    1.809490e-08
MAPK3     1.377238e-08
PIK3R3    1.220989e-08
SOS1      1.069349e-08
PIK3CD    1.062892e-08
              ...     
FANCG     1.395660e-12
GCLC      1.229822e-12
PDE4A     8.144559e-13
TAS2R5    6.896128e-14
PTGDR     6.622651e-14
Length: 4500, dtype: float64

In [125]:
cmn_feature_importance_df.mean().sort_values(ascending=False).index[:30]

Index(['PIK3R1', 'MAPK3', 'PIK3R3', 'SOS1', 'PIK3CD', 'PIK3CG', 'PIK3CB',
       'PRKCB', 'ALDH7A1', 'BRAF', 'MAPK1', 'RAC1', 'MAP2K1', 'PRKACA',
       'GNG12', 'PIK3CA', 'MAP2K2', 'GCDH', 'CXCL9', 'PRKCA', 'STAT3',
       'ITGA10', 'PDGFRA', 'GNB5', 'RIPK1', 'ADCY4', 'CCL5', 'PDGFC', 'ATF6B',
       'ARAF'],
      dtype='object')

In [126]:
male_feature_importance_df.mean().sort_values(ascending=False)

TUBB6       1.263241e-08
GRB2        1.184412e-08
MAP2K2      1.043119e-08
TUBB8       9.857849e-09
AKT3        9.741629e-09
                ...     
CHMP4B      1.368899e-12
PPP1R12B    1.160957e-12
ACER3       8.561765e-13
COX17       3.215596e-13
THOC2       5.834606e-14
Length: 4500, dtype: float64

In [127]:
male_feature_importance_df.mean().sort_values(ascending=False).index[:30]

Index(['TUBB6', 'GRB2', 'MAP2K2', 'TUBB8', 'AKT3', 'CREB3L4', 'PLCG2', 'ATF6B',
       'MYC', 'ARAF', 'KRAS', 'TUBB2A', 'RNF2', 'PIK3R2', 'TUBA4A', 'HADHA',
       'PIK3CD', 'NFATC2', 'PPP2R1A', 'WNT3', 'MAPK1', 'PIK3CA', 'TUBA8',
       'RELA', 'PRKCA', 'PPP3CB', 'PIK3CG', 'TUBA1C', 'AKT1', 'PIK3CB'],
      dtype='object')

In [128]:
female_feature_importance_df.mean().sort_values(ascending=False)

ADCY4     6.640382e-09
HRAS      6.415133e-09
RAF1      6.157602e-09
PIK3CB    6.081607e-09
PPP3CA    5.940696e-09
              ...     
PQBP1     7.210299e-13
PXMP2     6.814006e-13
EPHA7     6.785954e-13
B2M       5.501921e-13
LARS1     3.255750e-13
Length: 4500, dtype: float64

In [129]:
female_feature_importance_df.mean().sort_values(ascending=False).index[:30]

Index(['ADCY4', 'HRAS', 'RAF1', 'PIK3CB', 'PPP3CA', 'GNAS', 'ARAF', 'MAPK9',
       'LYN', 'CREB1', 'CHP1', 'CREB3L4', 'TUBA4A', 'PRKCA', 'CACNA1D', 'EGFR',
       'GNG11', 'EGF', 'ALDH2', 'NLK', 'PLCB1', 'TP53', 'SOS1', 'MAP2K2',
       'MAP2K1', 'SRC', 'ATF2', 'PIK3CG', 'CREB3L1', 'TUBA1A'],
      dtype='object')

In [130]:
np.intersect1d(cmn_feature_importance_df.mean().sort_values(ascending=False).index[:30], male_feature_importance_df.mean().sort_values(ascending=False).index[:30])

array(['ARAF', 'ATF6B', 'MAP2K2', 'MAPK1', 'PIK3CA', 'PIK3CB', 'PIK3CD',
       'PIK3CG', 'PRKCA'], dtype=object)

In [131]:
np.intersect1d(cmn_feature_importance_df.mean().sort_values(ascending=False).index[:30], female_feature_importance_df.mean().sort_values(ascending=False).index[:30])

array(['ADCY4', 'ARAF', 'MAP2K1', 'MAP2K2', 'PIK3CB', 'PIK3CG', 'PRKCA',
       'SOS1'], dtype=object)

In [132]:
np.intersect1d(male_feature_importance_df.mean().sort_values(ascending=False).index[:30], female_feature_importance_df.mean().sort_values(ascending=False).index[:30])

array(['ARAF', 'CREB3L4', 'MAP2K2', 'PIK3CB', 'PIK3CG', 'PRKCA', 'TUBA4A'],
      dtype=object)

---

In [133]:
np.intersect1d(cmn_feature_importance_df.mean().sort_values(ascending=False).index[:20], male_feature_importance_df.mean().sort_values(ascending=False).index[:20])

array(['MAP2K2', 'PIK3CD'], dtype=object)

In [134]:
np.intersect1d(cmn_feature_importance_df.mean().sort_values(ascending=False).index[:20], female_feature_importance_df.mean().sort_values(ascending=False).index[:20])

array(['PIK3CB', 'PRKCA'], dtype=object)

In [135]:
np.intersect1d(male_feature_importance_df.mean().sort_values(ascending=False).index[:20], female_feature_importance_df.mean().sort_values(ascending=False).index[:20])

array(['ARAF', 'CREB3L4', 'TUBA4A'], dtype=object)

---

# 1-sample sex-specific t-test compared with zero

---

In [136]:
from scipy.stats import ttest_1samp
from scipy.stats import ttest_ind
from datetime import datetime
from statsmodels.stats.multitest import multipletests

In [137]:
cmn_feature_importance_df

,DPM1,FGR,CFH,FUCA2,GCLC,NFYA,SEMA3F,CFTR,CYP51A1,RAD52,...,H2AC17,H4C2,H3C10,ADORA3,PIGY,H3C2,H3C3,NPBWR1,UGT1A3,UGT1A5
0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2,0.000000e+00,1.297577e-08,1.871284e-09,3.843207e-10,0.0,0.000000e+00,1.586767e-09,1.242851e-10,0.000000e+00,5.739490e-10,...,2.729920e-09,1.672082e-09,4.708784e-10,1.042283e-10,0.000000e+00,3.993251e-09,3.887306e-09,8.456346e-11,1.784719e-08,1.582071e-08
3,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
4,8.927430e-09,0.000000e+00,0.000000e+00,1.346626e-10,0.0,1.091458e-08,0.000000e+00,0.000000e+00,6.107713e-09,8.839507e-09,...,0.000000e+00,0.000000e+00,3.788405e-09,0.000000e+00,2.890757e-09,6.257296e-09,1.310120e-09,0.000000e+00,2.575537e-09,4.449813e-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
532,0.000000e+00,0.000000e+00,0.000000e+00,1.357786e-09,0.0,4.472802e-09,0.000000e+00,1.017437e-11,0.000000e+00,0.000000e+00,...,3.434133e-09,2.103414e-09,2.073065e-09,7.413477e-09,0.000000e+00,3.421358e-09,3.968318e-09,6.014770e-09,8.019709e-10,4.466618e-09
533,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
534,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [138]:
common_gene_pvalues = []
male_gene_pvalues = []
female_gene_pvalues = []
for idx in range(len(cmn_feature_importance_df.columns)):
    common_gene_pvalues.append(ttest_1samp(cmn_feature_importance_df.iloc[:, idx], popmean=0))
    male_gene_pvalues.append(ttest_1samp(male_feature_importance_df.iloc[:, idx], popmean=0))
    female_gene_pvalues.append(ttest_1samp(female_feature_importance_df.iloc[:, idx], popmean=0))

In [139]:
common_pathway_pvalues = []
male_pathway_pvalues = []
female_pathway_pvalues = []
for idx in range(len(cmn_pathway_importance_df.columns)):
    common_pathway_pvalues.append(ttest_1samp(cmn_pathway_importance_df.iloc[:, idx], popmean=0))
    male_pathway_pvalues.append(ttest_1samp(male_pathway_importance_df.iloc[:, idx], popmean=0))
    female_pathway_pvalues.append(ttest_1samp(female_pathway_importance_df.iloc[:, idx], popmean=0))

---

### Common

In [140]:
common_gene_pval_df = pd.DataFrame(common_gene_pvalues).rename(columns = {"pvalue" : "common_pval"})
common_gene_pval_df

,statistic,common_pval
0,6.079989,2.287627e-09
1,9.380847,1.848442e-19
2,7.258475,1.386100e-12
3,7.604117,1.297338e-13
4,8.005210,7.468135e-15
...,...,...
4495,11.189845,2.913395e-26
4496,9.035209,2.975707e-18
4497,5.207112,2.740145e-07
4498,10.378476,4.088994e-23


In [141]:
common_gene_pval_df["-log10(common_pval)"] = -np.log10(common_gene_pval_df["common_pval"])

In [142]:
common_gene_pval_df["Gene"] = cmn_feature_importance_df.columns.values

In [143]:
cmn_feature_importance_df.abs().mean().values

array([6.47010874e-10, 2.85370952e-09, 5.35051670e-10, ...,
       3.08477082e-10, 3.82363848e-09, 2.99019315e-09])

In [144]:
common_gene_pval_df["ABS_Importance"] = cmn_feature_importance_df.abs().mean().values
common_gene_pval_df

,statistic,common_pval,-log10(common_pval),Gene,ABS_Importance
0,6.079989,2.287627e-09,8.640615,DPM1,6.470109e-10
1,9.380847,1.848442e-19,18.733194,FGR,2.853710e-09
2,7.258475,1.386100e-12,11.858206,CFH,5.350517e-10
3,7.604117,1.297338e-13,12.886947,FUCA2,7.120090e-11
4,8.005210,7.468135e-15,14.126788,GCLC,1.229822e-12
...,...,...,...,...,...
4495,11.189845,2.913395e-26,25.535601,H3C2,1.318210e-09
4496,9.035209,2.975707e-18,17.526410,H3C3,7.361822e-10
4497,5.207112,2.740145e-07,6.562226,NPBWR1,3.084771e-10
4498,10.378476,4.088994e-23,22.388383,UGT1A3,3.823638e-09


In [145]:
common_gene_pval_df["Importance"] = cmn_feature_importance_df.mean().values
common_gene_pval_df

,statistic,common_pval,-log10(common_pval),Gene,ABS_Importance,Importance
0,6.079989,2.287627e-09,8.640615,DPM1,6.470109e-10,6.470109e-10
1,9.380847,1.848442e-19,18.733194,FGR,2.853710e-09,2.853710e-09
2,7.258475,1.386100e-12,11.858206,CFH,5.350517e-10,5.350517e-10
3,7.604117,1.297338e-13,12.886947,FUCA2,7.120090e-11,7.120090e-11
4,8.005210,7.468135e-15,14.126788,GCLC,1.229822e-12,1.229822e-12
...,...,...,...,...,...,...
4495,11.189845,2.913395e-26,25.535601,H3C2,1.318210e-09,1.318210e-09
4496,9.035209,2.975707e-18,17.526410,H3C3,7.361822e-10,7.361822e-10
4497,5.207112,2.740145e-07,6.562226,NPBWR1,3.084771e-10,3.084771e-10
4498,10.378476,4.088994e-23,22.388383,UGT1A3,3.823638e-09,3.823638e-09


---

In [146]:
common_pathway_pval_df = pd.DataFrame(common_pathway_pvalues).rename(columns = {"pvalue" : "common_pval"})
common_pathway_pval_df

,statistic,common_pval
0,12.309069,7.832699e-31
1,9.602434,2.996273e-20
2,9.020258,3.350149e-18
3,12.187231,2.531560e-30
4,12.660265,2.570743e-32
...,...,...
390,13.003385,8.694028e-34
391,9.963778,1.448963e-21
392,13.309182,4.088855e-35
393,10.789378,1.087196e-24


In [147]:
common_pathway_pval_df["-log10(common_pval)"] = -np.log10(common_pathway_pval_df["common_pval"])

In [148]:
common_pathway_pval_df["pathway"] = cmn_pathway_importance_df.columns.values

In [149]:
common_pathway_pval_df["ABS_Importance"] = cmn_pathway_importance_df.abs().mean().values
common_pathway_pval_df

,statistic,common_pval,-log10(common_pval),pathway,ABS_Importance
0,12.309069,7.832699e-31,30.106089,KEGG_ABC_TRANSPORTERS,2.338675e-07
1,9.602434,2.996273e-20,19.523419,KEGG_ACUTE_MYELOID_LEUKEMIA,1.990231e-07
2,9.020258,3.350149e-18,17.474936,KEGG_ADHERENS_JUNCTION,2.715652e-07
3,12.187231,2.531560e-30,29.596612,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,2.138752e-07
4,12.660265,2.570743e-32,31.589941,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,2.471272e-07
...,...,...,...,...,...
390,13.003385,8.694028e-34,33.060779,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,2.473844e-07
391,9.963778,1.448963e-21,20.838943,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,1.097209e-07
392,13.309182,4.088855e-35,34.388398,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,1.135037e-07
393,10.789378,1.087196e-24,23.963692,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,2.278621e-07


In [150]:
common_pathway_pval_df["Importance"] = cmn_pathway_importance_df.mean().values
common_pathway_pval_df

,statistic,common_pval,-log10(common_pval),pathway,ABS_Importance,Importance
0,12.309069,7.832699e-31,30.106089,KEGG_ABC_TRANSPORTERS,2.338675e-07,2.338675e-07
1,9.602434,2.996273e-20,19.523419,KEGG_ACUTE_MYELOID_LEUKEMIA,1.990231e-07,1.990231e-07
2,9.020258,3.350149e-18,17.474936,KEGG_ADHERENS_JUNCTION,2.715652e-07,2.715652e-07
3,12.187231,2.531560e-30,29.596612,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,2.138752e-07,2.138752e-07
4,12.660265,2.570743e-32,31.589941,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,2.471272e-07,2.471272e-07
...,...,...,...,...,...,...
390,13.003385,8.694028e-34,33.060779,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,2.473844e-07,2.473844e-07
391,9.963778,1.448963e-21,20.838943,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,1.097209e-07,1.097209e-07
392,13.309182,4.088855e-35,34.388398,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,1.135037e-07,1.135037e-07
393,10.789378,1.087196e-24,23.963692,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,2.278621e-07,2.278621e-07


---

---

### Male

In [151]:
male_gene_pval_df = pd.DataFrame(male_gene_pvalues).rename(columns = {"pvalue" : "male_pval"})
male_gene_pval_df

,statistic,male_pval
0,4.420662,1.594607e-05
1,3.880689,1.403774e-04
2,4.178111,4.349354e-05
3,5.254186,3.715217e-07
4,5.164529,5.691715e-07
...,...,...
4495,6.024979,7.732073e-09
4496,5.634714,5.744080e-08
4497,1.824526,6.952824e-02
4498,5.628402,5.929036e-08


In [152]:
male_gene_pval_df["-log10(male_pval)"] = -np.log10(male_gene_pval_df["male_pval"])

In [153]:
male_gene_pval_df["Gene"] = male_feature_importance_df.columns.values

In [154]:
male_gene_pval_df["ABS_Importance"] = male_feature_importance_df.abs().mean().values
male_gene_pval_df

,statistic,male_pval,-log10(male_pval),Gene,ABS_Importance
0,4.420662,1.594607e-05,4.797346,DPM1,1.213664e-09
1,3.880689,1.403774e-04,3.852703,FGR,6.224591e-10
2,4.178111,4.349354e-05,4.361575,CFH,1.020769e-09
3,5.254186,3.715217e-07,6.430016,FUCA2,5.451535e-10
4,5.164529,5.691715e-07,6.244757,GCLC,5.423148e-11
...,...,...,...,...,...
4495,6.024979,7.732073e-09,8.111704,H3C2,5.614533e-10
4496,5.634714,5.744080e-08,7.240779,H3C3,3.226365e-10
4497,1.824526,6.952824e-02,1.157839,NPBWR1,3.649776e-10
4498,5.628402,5.929036e-08,7.227016,UGT1A3,1.180480e-09


In [155]:
male_gene_pval_df["Importance"] = male_feature_importance_df.mean().values
male_gene_pval_df

,statistic,male_pval,-log10(male_pval),Gene,ABS_Importance,Importance
0,4.420662,1.594607e-05,4.797346,DPM1,1.213664e-09,1.213664e-09
1,3.880689,1.403774e-04,3.852703,FGR,6.224591e-10,6.224591e-10
2,4.178111,4.349354e-05,4.361575,CFH,1.020769e-09,1.020769e-09
3,5.254186,3.715217e-07,6.430016,FUCA2,5.451535e-10,5.451535e-10
4,5.164529,5.691715e-07,6.244757,GCLC,5.423148e-11,5.423148e-11
...,...,...,...,...,...,...
4495,6.024979,7.732073e-09,8.111704,H3C2,5.614533e-10,5.614533e-10
4496,5.634714,5.744080e-08,7.240779,H3C3,3.226365e-10,3.226365e-10
4497,1.824526,6.952824e-02,1.157839,NPBWR1,3.649776e-10,3.649776e-10
4498,5.628402,5.929036e-08,7.227016,UGT1A3,1.180480e-09,1.180480e-09


---

In [156]:
male_pathway_pval_df = pd.DataFrame(male_pathway_pvalues).rename(columns = {"pvalue" : "male_pval"})
male_pathway_pval_df

,statistic,male_pval
0,7.258364,7.957114e-12
1,6.885783,6.888663e-11
2,6.856594,8.135071e-11
3,3.423629,7.460943e-04
4,7.461728,2.385123e-12
...,...,...
390,7.777268,3.551454e-13
391,6.747892,1.505712e-10
392,6.954740,4.642913e-11
393,6.276555,2.026891e-09


In [157]:
male_pathway_pval_df["-log10(male_pval)"] = -np.log10(male_pathway_pval_df["male_pval"])

In [158]:
male_pathway_pval_df["pathway"] = male_pathway_importance_df.columns.values

In [159]:
male_pathway_pval_df["ABS_Importance"] = male_pathway_importance_df.abs().mean().values
male_pathway_pval_df

,statistic,male_pval,-log10(male_pval),pathway,ABS_Importance
0,7.258364,7.957114e-12,11.099244,KEGG_ABC_TRANSPORTERS,1.815041e-07
1,6.885783,6.888663e-11,10.161865,KEGG_ACUTE_MYELOID_LEUKEMIA,3.460061e-07
2,6.856594,8.135071e-11,10.089639,KEGG_ADHERENS_JUNCTION,1.562941e-07
3,3.423629,7.460943e-04,3.127206,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,1.015440e-07
4,7.461728,2.385123e-12,11.622489,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,1.745154e-07
...,...,...,...,...,...
390,7.777268,3.551454e-13,12.449594,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,7.642845e-08
391,6.747892,1.505712e-10,9.822258,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,1.640847e-07
392,6.954740,4.642913e-11,10.333209,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,8.437033e-08
393,6.276555,2.026891e-09,8.693170,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,8.647754e-08


In [160]:
male_pathway_pval_df["Importance"] = male_pathway_importance_df.mean().values
male_pathway_pval_df

,statistic,male_pval,-log10(male_pval),pathway,ABS_Importance,Importance
0,7.258364,7.957114e-12,11.099244,KEGG_ABC_TRANSPORTERS,1.815041e-07,1.815041e-07
1,6.885783,6.888663e-11,10.161865,KEGG_ACUTE_MYELOID_LEUKEMIA,3.460061e-07,3.460061e-07
2,6.856594,8.135071e-11,10.089639,KEGG_ADHERENS_JUNCTION,1.562941e-07,1.562941e-07
3,3.423629,7.460943e-04,3.127206,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,1.015440e-07,1.015440e-07
4,7.461728,2.385123e-12,11.622489,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,1.745154e-07,1.745154e-07
...,...,...,...,...,...,...
390,7.777268,3.551454e-13,12.449594,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,7.642845e-08,7.642845e-08
391,6.747892,1.505712e-10,9.822258,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,1.640847e-07,1.640847e-07
392,6.954740,4.642913e-11,10.333209,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,8.437033e-08,8.437033e-08
393,6.276555,2.026891e-09,8.693170,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,8.647754e-08,8.647754e-08


---

---

### Female

In [161]:
female_gene_pval_df = pd.DataFrame(female_gene_pvalues).rename(columns = {"pvalue" : "female_pval"})
female_gene_pval_df

,statistic,female_pval
0,8.673910,1.948487e-16
1,7.116079,6.992246e-12
2,6.225512,1.462388e-09
3,4.916005,1.396492e-06
4,5.799151,1.564174e-08
...,...,...
4495,6.908518,2.542321e-11
4496,6.928838,2.243180e-11
4497,3.558765,4.274278e-04
4498,9.282109,2.303009e-18


In [162]:
female_gene_pval_df["-log10(female_pval)"] = -np.log10(female_gene_pval_df["female_pval"])

In [163]:
female_gene_pval_df["Gene"] = female_feature_importance_df.columns.values

In [164]:
female_gene_pval_df["ABS_Importance"] = female_feature_importance_df.abs().mean().values
female_gene_pval_df

,statistic,female_pval,-log10(female_pval),Gene,ABS_Importance
0,8.673910,1.948487e-16,15.710303,DPM1,5.373825e-10
1,7.116079,6.992246e-12,11.155383,FGR,3.582191e-10
2,6.225512,1.462388e-09,8.834937,CFH,1.606058e-10
3,4.916005,1.396492e-06,5.854962,FUCA2,2.964355e-10
4,5.799151,1.564174e-08,7.805715,GCLC,2.914352e-12
...,...,...,...,...,...
4495,6.908518,2.542321e-11,10.594770,H3C2,7.897685e-10
4496,6.928838,2.243180e-11,10.649136,H3C3,4.399401e-10
4497,3.558765,4.274278e-04,3.369137,NPBWR1,1.520569e-10
4498,9.282109,2.303009e-18,17.637704,UGT1A3,2.237039e-09


In [165]:
female_gene_pval_df["Importance"] = female_feature_importance_df.mean().values
female_gene_pval_df

,statistic,female_pval,-log10(female_pval),Gene,ABS_Importance,Importance
0,8.673910,1.948487e-16,15.710303,DPM1,5.373825e-10,5.373825e-10
1,7.116079,6.992246e-12,11.155383,FGR,3.582191e-10,3.582191e-10
2,6.225512,1.462388e-09,8.834937,CFH,1.606058e-10,1.606058e-10
3,4.916005,1.396492e-06,5.854962,FUCA2,2.964355e-10,2.964355e-10
4,5.799151,1.564174e-08,7.805715,GCLC,2.914352e-12,2.914352e-12
...,...,...,...,...,...,...
4495,6.908518,2.542321e-11,10.594770,H3C2,7.897685e-10,7.897685e-10
4496,6.928838,2.243180e-11,10.649136,H3C3,4.399401e-10,4.399401e-10
4497,3.558765,4.274278e-04,3.369137,NPBWR1,1.520569e-10,1.520569e-10
4498,9.282109,2.303009e-18,17.637704,UGT1A3,2.237039e-09,2.237039e-09


---

In [166]:
female_pathway_pval_df = pd.DataFrame(female_pathway_pvalues).rename(columns = {"pvalue" : "female_pval"})
female_pathway_pval_df

,statistic,female_pval
0,6.786595,5.358602e-11
1,9.495095,4.675436e-19
2,8.398427,1.370513e-15
3,8.767094,9.985971e-17
4,8.402802,1.329107e-15
...,...,...
390,7.525195,5.083418e-13
391,7.950005,3.013529e-14
392,7.899946,4.226411e-14
393,8.305267,2.627518e-15


In [167]:
female_pathway_pval_df["-log10(female_pval)"] = -np.log10(female_pathway_pval_df["female_pval"])

In [168]:
female_pathway_pval_df["pathway"] = female_pathway_importance_df.columns.values

In [169]:
female_pathway_pval_df["ABS_Importance"] = female_pathway_importance_df.abs().mean().values
female_pathway_pval_df

,statistic,female_pval,-log10(female_pval),pathway,ABS_Importance
0,6.786595,5.358602e-11,10.270948,KEGG_ABC_TRANSPORTERS,7.227374e-08
1,9.495095,4.675436e-19,18.330178,KEGG_ACUTE_MYELOID_LEUKEMIA,1.276121e-07
2,8.398427,1.370513e-15,14.863117,KEGG_ADHERENS_JUNCTION,1.109605e-07
3,8.767094,9.985971e-17,16.000610,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,1.182161e-07
4,8.402802,1.329107e-15,14.876440,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,2.042175e-07
...,...,...,...,...,...
390,7.525195,5.083418e-13,12.293844,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,1.346889e-07
391,7.950005,3.013529e-14,13.520925,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,8.433279e-08
392,7.899946,4.226411e-14,13.374028,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,1.286432e-07
393,8.305267,2.627518e-15,14.580454,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,1.531703e-07


In [170]:
female_pathway_pval_df["Importance"] = female_pathway_importance_df.mean().values
female_pathway_pval_df

,statistic,female_pval,-log10(female_pval),pathway,ABS_Importance,Importance
0,6.786595,5.358602e-11,10.270948,KEGG_ABC_TRANSPORTERS,7.227374e-08,7.227374e-08
1,9.495095,4.675436e-19,18.330178,KEGG_ACUTE_MYELOID_LEUKEMIA,1.276121e-07,1.276121e-07
2,8.398427,1.370513e-15,14.863117,KEGG_ADHERENS_JUNCTION,1.109605e-07,1.109605e-07
3,8.767094,9.985971e-17,16.000610,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,1.182161e-07,1.182161e-07
4,8.402802,1.329107e-15,14.876440,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,2.042175e-07,2.042175e-07
...,...,...,...,...,...,...
390,7.525195,5.083418e-13,12.293844,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,1.346889e-07,1.346889e-07
391,7.950005,3.013529e-14,13.520925,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,8.433279e-08,8.433279e-08
392,7.899946,4.226411e-14,13.374028,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,1.286432e-07,1.286432e-07
393,8.305267,2.627518e-15,14.580454,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,1.531703e-07,1.531703e-07


---

---

# FDR-correction

---

---

# BH approach - controlling FDR

---

## Significant Genes

---

In [171]:
male_multi_test = multipletests(male_gene_pval_df["male_pval"], alpha=0.01, method='fdr_bh')

In [172]:
male_multi_test

(array([ True,  True,  True, ..., False,  True,  True]),
 array([2.27801068e-05, 1.67692661e-04, 5.76497591e-05, ...,
        6.95282375e-02, 2.56876997e-07, 1.61178328e-08]),
 2.2334054733397224e-06,
 2.222222222222222e-06)

In [173]:
male_gene_pval_df["FDR_BH"] = male_multi_test[1]
male_gene_pval_df["-log10(FDR_BH)"] = -np.log10(male_gene_pval_df["FDR_BH"])

In [174]:
male_gene_pval_df

,statistic,male_pval,-log10(male_pval),Gene,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
0,4.420662,1.594607e-05,4.797346,DPM1,1.213664e-09,1.213664e-09,2.278011e-05,4.642444
1,3.880689,1.403774e-04,3.852703,FGR,6.224591e-10,6.224591e-10,1.676927e-04,3.775486
2,4.178111,4.349354e-05,4.361575,CFH,1.020769e-09,1.020769e-09,5.764976e-05,4.239203
3,5.254186,3.715217e-07,6.430016,FUCA2,5.451535e-10,5.451535e-10,1.026933e-06,5.988458
4,5.164529,5.691715e-07,6.244757,GCLC,5.423148e-11,5.423148e-11,1.465259e-06,5.834086
...,...,...,...,...,...,...,...,...
4495,6.024979,7.732073e-09,8.111704,H3C2,5.614533e-10,5.614533e-10,7.029158e-08,7.153097
4496,5.634714,5.744080e-08,7.240779,H3C3,3.226365e-10,3.226365e-10,2.511989e-07,6.599982
4497,1.824526,6.952824e-02,1.157839,NPBWR1,3.649776e-10,3.649776e-10,6.952824e-02,1.157839
4498,5.628402,5.929036e-08,7.227016,UGT1A3,1.180480e-09,1.180480e-09,2.568770e-07,6.590275


In [181]:
male_gene_pval_df.to_csv(f"[{date}_{num}]{asthma}_Male_Genes_p-values_FDR_Correction.csv", header = True, index = False)

---

In [176]:
female_multi_test = multipletests(female_gene_pval_df["female_pval"], alpha=0.01, method='fdr_bh')

In [177]:
female_multi_test

(array([ True,  True,  True, ...,  True,  True,  True]),
 array([4.29813225e-15, 1.81669212e-11, 2.47675865e-09, ...,
        4.38137827e-04, 2.75598750e-16, 9.84634656e-15]),
 2.2334054733397224e-06,
 2.222222222222222e-06)

In [178]:
female_gene_pval_df["FDR_BH"] = female_multi_test[1]
female_gene_pval_df["-log10(FDR_BH)"] = -np.log10(female_gene_pval_df["FDR_BH"])

In [179]:
female_gene_pval_df

,statistic,female_pval,-log10(female_pval),Gene,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
0,8.673910,1.948487e-16,15.710303,DPM1,5.373825e-10,5.373825e-10,4.298132e-15,14.366720
1,7.116079,6.992246e-12,11.155383,FGR,3.582191e-10,3.582191e-10,1.816692e-11,10.740719
2,6.225512,1.462388e-09,8.834937,CFH,1.606058e-10,1.606058e-10,2.476759e-09,8.606116
3,4.916005,1.396492e-06,5.854962,FUCA2,2.964355e-10,2.964355e-10,1.619642e-06,5.790581
4,5.799151,1.564174e-08,7.805715,GCLC,2.914352e-12,2.914352e-12,2.216939e-08,7.654246
...,...,...,...,...,...,...,...,...
4495,6.908518,2.542321e-11,10.594770,H3C2,7.897685e-10,7.897685e-10,5.734558e-11,10.241500
4496,6.928838,2.243180e-11,10.649136,H3C3,4.399401e-10,4.399401e-10,5.108457e-11,10.291710
4497,3.558765,4.274278e-04,3.369137,NPBWR1,1.520569e-10,1.520569e-10,4.381378e-04,3.358389
4498,9.282109,2.303009e-18,17.637704,UGT1A3,2.237039e-09,2.237039e-09,2.755987e-16,15.559723


In [182]:
female_gene_pval_df.to_csv(f"[{date}_{num}]{asthma}_Female_Genes_p-values_FDR_Correction.csv", header = True, index = False)

---

In [183]:
common_multi_test = multipletests(common_gene_pval_df["common_pval"], alpha=0.01, method='fdr_bh')

In [184]:
common_multi_test

(array([ True,  True,  True, ...,  True,  True,  True]),
 array([2.52003000e-09, 5.91186271e-19, 1.81796815e-12, ...,
        2.83593639e-07, 3.03638188e-22, 7.87331106e-22]),
 2.2334054733397224e-06,
 2.222222222222222e-06)

In [185]:
common_gene_pval_df["FDR_BH"] = common_multi_test[1]
common_gene_pval_df["-log10(FDR_BH)"] = -np.log10(common_gene_pval_df["FDR_BH"])

In [186]:
common_gene_pval_df

,statistic,common_pval,-log10(common_pval),Gene,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
0,6.079989,2.287627e-09,8.640615,DPM1,6.470109e-10,6.470109e-10,2.520030e-09,8.598594
1,9.380847,1.848442e-19,18.733194,FGR,2.853710e-09,2.853710e-09,5.911863e-19,18.228276
2,7.258475,1.386100e-12,11.858206,CFH,5.350517e-10,5.350517e-10,1.817968e-12,11.740414
3,7.604117,1.297338e-13,12.886947,FUCA2,7.120090e-11,7.120090e-11,1.835856e-13,12.736161
4,8.005210,7.468135e-15,14.126788,GCLC,1.229822e-12,1.229822e-12,1.192993e-14,13.923362
...,...,...,...,...,...,...,...,...
4495,11.189845,2.913395e-26,25.535601,H3C2,1.318210e-09,1.318210e-09,7.757559e-25,24.110275
4496,9.035209,2.975707e-18,17.526410,H3C3,7.361822e-10,7.361822e-10,7.544046e-18,17.122396
4497,5.207112,2.740145e-07,6.562226,NPBWR1,3.084771e-10,3.084771e-10,2.835936e-07,6.547304
4498,10.378476,4.088994e-23,22.388383,UGT1A3,3.823638e-09,3.823638e-09,3.036382e-22,21.517644


In [188]:
common_gene_pval_df.to_csv(f"[{date}_{num}]{asthma}_Common_Genes_p-values_FDR_Correction.csv", header = True, index = False)

---

In [189]:
common_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)

,statistic,common_pval,-log10(common_pval),Gene,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
135,12.404758,3.103402e-31,30.508162,IGF1,3.370056e-09,3.370056e-09,1.161412e-27,26.935014
389,12.352220,5.161831e-31,30.287196,NCK2,3.132305e-09,3.132305e-09,1.161412e-27,26.935014
344,12.191960,2.419178e-30,29.616332,TNFRSF1A,4.439549e-09,4.439549e-09,2.721575e-27,26.565180
3364,12.215020,1.938341e-30,29.712570,IRS1,3.636946e-09,3.636946e-09,2.721575e-27,26.565180
817,12.152969,3.516872e-30,29.453843,PCK2,3.179476e-09,3.179476e-09,3.165185e-27,26.499601
...,...,...,...,...,...,...,...,...
2195,4.284041,2.176316e-05,4.662278,CTSV,2.273992e-10,2.273992e-10,2.177283e-05,4.662085
833,4.284041,2.176316e-05,4.662278,CTSZ,1.397572e-11,1.397572e-11,2.177283e-05,4.662085
4016,4.284041,2.176316e-05,4.662278,CLN3,4.434384e-10,4.434384e-10,2.177283e-05,4.662085
3177,4.276265,2.251087e-05,4.647608,INPPL1,6.456289e-10,6.456289e-10,2.251588e-05,4.647511


In [190]:
male_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)

,statistic,male_pval,-log10(male_pval),Gene,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
2486,7.633609,8.495851e-13,12.070793,WNT9A,3.724897e-09,3.724897e-09,3.823133e-09,8.417581
4241,7.353856,4.529477e-12,11.343952,HSPA1A,1.186650e-09,1.186650e-09,7.136924e-09,8.146489
2204,7.296845,6.343933e-12,11.197641,DCTN3,1.866009e-09,1.866009e-09,7.136924e-09,8.146489
4378,7.304914,6.049068e-12,11.218312,PLA2G4B,2.245191e-09,2.245191e-09,7.136924e-09,8.146489
4210,7.127267,1.713132e-11,10.766209,DMD,9.760764e-10,9.760764e-10,7.402253e-09,8.130636
...,...,...,...,...,...,...,...,...
3480,1.824526,6.952824e-02,1.157839,PTGER4,2.784970e-10,2.784970e-10,6.952824e-02,1.157839
3653,1.824526,6.952824e-02,1.157839,P2RY2,2.159436e-10,2.159436e-10,6.952824e-02,1.157839
1346,1.824526,6.952824e-02,1.157839,NR3C1,1.413187e-11,1.413187e-11,6.952824e-02,1.157839
3125,1.824526,6.952824e-02,1.157839,GRIK2,4.521905e-11,4.521905e-11,6.952824e-02,1.157839


In [191]:
female_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)

,statistic,female_pval,-log10(female_pval),Gene,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
1655,9.656465,1.378589e-19,18.860565,PIK3CA,3.740809e-09,3.740809e-09,6.892944e-17,16.161595
3647,9.667827,1.264472e-19,18.898091,LPL,1.016471e-09,1.016471e-09,6.892944e-17,16.161595
3486,9.746548,6.938416e-20,19.158740,PIK3CD,3.652380e-09,3.652380e-09,6.892944e-17,16.161595
3148,9.688575,1.079747e-19,18.966678,CDK5,2.413308e-09,2.413308e-09,6.892944e-17,16.161595
3872,9.731079,7.808527e-20,19.107431,IRAK1,2.276214e-09,2.276214e-09,6.892944e-17,16.161595
...,...,...,...,...,...,...,...,...
1223,2.295872,2.231163e-02,1.651469,CARS1,1.270084e-11,1.270084e-11,2.231163e-02,1.651469
1189,2.295872,2.231163e-02,1.651469,SEPSECS,6.852217e-11,6.852217e-11,2.231163e-02,1.651469
1733,2.295872,2.231163e-02,1.651469,AARS2,4.880392e-11,4.880392e-11,2.231163e-02,1.651469
1056,2.295872,2.231163e-02,1.651469,GARS1,6.331410e-11,6.331410e-11,2.231163e-02,1.651469


In [192]:
common_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["Gene"].iloc[:15].values

array(['IGF1', 'NCK2', 'TNFRSF1A', 'IRS1', 'PCK2', 'UGT1A8', 'PRKAA1',
       'ESPL1', 'PIK3CG', 'SHC2', 'MGST1', 'CREB3L1', 'BIRC3', 'POLA1',
       'IRAK1'], dtype=object)

In [193]:
male_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["Gene"].iloc[:15].values

array(['WNT9A', 'HSPA1A', 'DCTN3', 'PLA2G4B', 'DMD', 'MECOM', 'CREB3',
       'PTCH1', 'PINK1', 'SMO', 'ITGA7', 'MTOR', 'HSPA8', 'GSTM3',
       'NDUFB1'], dtype=object)

In [194]:
female_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["Gene"].iloc[:15].values

array(['PIK3CA', 'LPL', 'PIK3CD', 'CDK5', 'IRAK1', 'PIK3CB', 'SMAD3',
       'CRKL', 'CDC45', 'TRAF6', 'FAS', 'PDGFB', 'CDKN1A', 'TUBB8',
       'CXCR4'], dtype=object)

In [195]:
np.intersect1d(common_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["Gene"].iloc[:15].values, male_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["Gene"].iloc[:15].values)

array([], dtype=object)

In [196]:
np.intersect1d(common_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["Gene"].iloc[:15].values, female_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["Gene"].iloc[:15].values)

array(['IRAK1'], dtype=object)

In [197]:
np.intersect1d(male_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["Gene"].iloc[:15].values, female_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["Gene"].iloc[:15].values)

array([], dtype=object)

---

---

---

In [198]:
significant_cmn_gene_df = cmn_gene_df[['IGF1', 'NCK2', 'TNFRSF1A', 'IRS1', 'PCK2', 'UGT1A8', 'PRKAA1',
       'ESPL1', 'PIK3CG', 'SHC2', 'MGST1', 'CREB3L1', 'BIRC3', 'POLA1',
       'IRAK1']]
significant_cmn_gene_df

,IGF1,NCK2,TNFRSF1A,IRS1,PCK2,UGT1A8,PRKAA1,ESPL1,PIK3CG,SHC2,MGST1,CREB3L1,BIRC3,POLA1,IRAK1
0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2,8.896932e-09,1.248504e-08,1.359194e-08,-1.145992e-08,-6.510054e-09,-1.698384e-08,3.101509e-09,-9.146376e-09,3.992320e-08,-1.343938e-08,7.905642e-09,2.414391e-08,6.425935e-09,-1.578491e-08,1.067880e-08
3,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
4,8.024725e-09,4.635266e-09,-1.536026e-09,1.695587e-09,-5.632639e-09,-1.798840e-08,-3.341918e-09,1.926395e-09,-3.171982e-09,5.286234e-09,-2.175612e-08,3.132361e-08,-4.527978e-10,2.278959e-08,-3.338734e-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
532,9.503807e-10,4.434353e-09,-1.590355e-08,1.806717e-08,2.766460e-09,1.747780e-09,-1.321082e-09,-4.406156e-09,1.110969e-09,2.437925e-09,-2.129977e-08,1.471741e-08,3.697003e-10,-2.008735e-09,4.210885e-09
533,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
534,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [199]:
np.sign(significant_cmn_gene_df).apply(pd.Series.value_counts)

,IGF1,NCK2,TNFRSF1A,IRS1,PCK2,UGT1A8,PRKAA1,ESPL1,PIK3CG,SHC2,MGST1,CREB3L1,BIRC3,POLA1,IRAK1
-1.0,39,39,38,129,133,121,42,126,32,127,58,41,22,121,77
0.0,370,378,371,371,371,372,377,371,370,372,371,370,373,373,370
1.0,127,119,127,36,32,43,117,39,134,37,107,125,141,42,89


In [200]:
significant_male_gene_df = male_gene_df[['WNT9A', 'HSPA1A', 'DCTN3', 'PLA2G4B', 'DMD', 'MECOM', 'CREB3',
       'PTCH1', 'PINK1', 'SMO', 'ITGA7', 'MTOR', 'HSPA8', 'GSTM3',
       'NDUFB1']]
significant_male_gene_df

,WNT9A,HSPA1A,DCTN3,PLA2G4B,DMD,MECOM,CREB3,PTCH1,PINK1,SMO,ITGA7,MTOR,HSPA8,GSTM3,NDUFB1
0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2,2.055595e-08,4.275647e-09,-7.860919e-09,3.976928e-09,-4.612610e-09,-5.157283e-09,6.393835e-09,-2.163944e-08,-1.026648e-08,9.282475e-09,2.775959e-08,6.047089e-09,4.135434e-09,-2.148463e-09,-1.066042e-08
3,-1.716712e-08,7.517994e-09,-6.094806e-09,1.196302e-08,-3.747233e-09,-5.754178e-09,-1.354512e-08,1.388113e-08,-5.050135e-09,-8.424428e-09,-8.428849e-09,1.067338e-08,6.445603e-09,-4.077610e-09,-6.298714e-09
4,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
202,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
203,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
204,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [201]:
np.sign(significant_male_gene_df).apply(pd.Series.value_counts)

,WNT9A,HSPA1A,DCTN3,PLA2G4B,DMD,MECOM,CREB3,PTCH1,PINK1,SMO,ITGA7,MTOR,HSPA8,GSTM3,NDUFB1
-1.0,19,28,47,14,37,34,31,30,43,23,29,6,24,36,47
0.0,151,151,151,151,153,157,151,153,151,153,151,151,153,151,151
1.0,36,27,8,41,16,15,24,23,12,30,26,49,29,19,8


In [202]:
significant_female_gene_df = female_gene_df[['PIK3CA', 'LPL', 'PIK3CD', 'CDK5', 'IRAK1', 'PIK3CB', 'SMAD3',
       'CRKL', 'CDC45', 'TRAF6', 'FAS', 'PDGFB', 'CDKN1A', 'TUBB8',
       'CXCR4']]
significant_female_gene_df

,PIK3CA,LPL,PIK3CD,CDK5,IRAK1,PIK3CB,SMAD3,CRKL,CDC45,TRAF6,FAS,PDGFB,CDKN1A,TUBB8,CXCR4
0,-1.123966e-08,-1.214319e-09,-7.589385e-09,1.854076e-09,-1.079891e-08,-1.737913e-09,6.856331e-09,5.351788e-09,-4.981418e-09,4.340736e-09,-1.477606e-08,9.759102e-09,-3.497365e-09,-1.124376e-10,4.380431e-09
1,-1.808148e-08,4.600693e-09,-8.398409e-09,9.139160e-09,-6.986323e-10,-3.413144e-08,-1.412976e-08,3.585726e-09,2.936916e-09,8.092743e-09,-7.592718e-09,-8.516272e-09,-2.087631e-08,1.383750e-08,1.131482e-08
2,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
3,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
4,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
326,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
327,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
328,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [203]:
np.sign(significant_female_gene_df).apply(pd.Series.value_counts)

,PIK3CA,LPL,PIK3CD,CDK5,IRAK1,PIK3CB,SMAD3,CRKL,CDC45,TRAF6,FAS,PDGFB,CDKN1A,TUBB8,CXCR4
-1.0,47,74,48,59,34,45,83,78,42,68,36,73,93,55,32
0.0,225,234,225,225,225,225,225,225,226,225,225,225,225,225,226
1.0,58,22,57,46,71,60,22,27,62,37,69,32,12,50,72


---

In [401]:
pd.set_option("display.max_rows", 11)

---

---

---

## Significant Pathways

---

In [204]:
male_multi_test = multipletests(male_pathway_pval_df["male_pval"], alpha=0.01, method='fdr_bh')

In [206]:
male_pathway_pval_df["FDR_BH"] = male_multi_test[1]
male_pathway_pval_df["-log10(FDR_BH)"] = -np.log10(male_pathway_pval_df["FDR_BH"])

In [207]:
male_pathway_pval_df

,statistic,male_pval,-log10(male_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
0,7.258364,7.957114e-12,11.099244,KEGG_ABC_TRANSPORTERS,1.815041e-07,1.815041e-07,5.419069e-11,10.266075
1,6.885783,6.888663e-11,10.161865,KEGG_ACUTE_MYELOID_LEUKEMIA,3.460061e-07,3.460061e-07,2.015572e-10,9.695602
2,6.856594,8.135071e-11,10.089639,KEGG_ADHERENS_JUNCTION,1.562941e-07,1.562941e-07,2.259357e-10,9.646015
3,3.423629,7.460943e-04,3.127206,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,1.015440e-07,1.015440e-07,7.576022e-04,3.120559
4,7.461728,2.385123e-12,11.622489,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,1.745154e-07,1.745154e-07,3.755520e-11,10.425330
...,...,...,...,...,...,...,...,...
390,7.777268,3.551454e-13,12.449594,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,7.642845e-08,7.642845e-08,3.755520e-11,10.425330
391,6.747892,1.505712e-10,9.822258,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,1.640847e-07,1.640847e-07,3.498566e-10,9.456110
392,6.954740,4.642913e-11,10.333209,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,8.437033e-08,8.437033e-08,1.491017e-10,9.826517
393,6.276555,2.026891e-09,8.693170,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,8.647754e-08,8.647754e-08,2.976289e-09,8.526325


In [208]:
male_pathway_pval_df.to_csv(f"[{date}_{num}]{asthma}_Male_Pathways_p-values_FDR_Correction.csv", header = True, index = False)

---

In [209]:
female_multi_test = multipletests(female_pathway_pval_df["female_pval"], alpha=0.01, method='fdr_bh')

In [210]:
female_pathway_pval_df["FDR_BH"] = female_multi_test[1]
female_pathway_pval_df["-log10(FDR_BH)"] = -np.log10(female_pathway_pval_df["FDR_BH"])

In [211]:
female_pathway_pval_df

,statistic,female_pval,-log10(female_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
0,6.786595,5.358602e-11,10.270948,KEGG_ABC_TRANSPORTERS,7.227374e-08,7.227374e-08,6.337269e-11,10.198098
1,9.495095,4.675436e-19,18.330178,KEGG_ACUTE_MYELOID_LEUKEMIA,1.276121e-07,1.276121e-07,2.279997e-18,17.642066
2,8.398427,1.370513e-15,14.863117,KEGG_ADHERENS_JUNCTION,1.109605e-07,1.109605e-07,2.819545e-15,14.549821
3,8.767094,9.985971e-17,16.000610,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,1.182161e-07,1.182161e-07,2.629639e-16,15.580104
4,8.402802,1.329107e-15,14.876440,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,2.042175e-07,2.042175e-07,2.777763e-15,14.556305
...,...,...,...,...,...,...,...,...
390,7.525195,5.083418e-13,12.293844,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,1.346889e-07,1.346889e-07,7.095231e-13,12.149033
391,7.950005,3.013529e-14,13.520925,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,8.433279e-08,8.433279e-08,4.742406e-14,13.324001
392,7.899946,4.226411e-14,13.374028,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,1.286432e-07,1.286432e-07,6.624732e-14,13.178832
393,8.305267,2.627518e-15,14.580454,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,1.531703e-07,1.531703e-07,4.989529e-15,14.301940


In [213]:
female_pathway_pval_df.to_csv(f"[{date}_{num}]{asthma}_Female_Pathways_p-values_FDR_Correction.csv", header = True, index = False)

---

In [214]:
common_multi_test = multipletests(common_pathway_pval_df["common_pval"], alpha=0.01, method='fdr_bh')

In [215]:
common_pathway_pval_df["FDR_BH"] = common_multi_test[1]
common_pathway_pval_df["-log10(FDR_BH)"] = -np.log10(common_pathway_pval_df["FDR_BH"])

In [216]:
common_pathway_pval_df

,statistic,common_pval,-log10(common_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
0,12.309069,7.832699e-31,30.106089,KEGG_ABC_TRANSPORTERS,2.338675e-07,2.338675e-07,2.361768e-30,29.626763
1,9.602434,2.996273e-20,19.523419,KEGG_ACUTE_MYELOID_LEUKEMIA,1.990231e-07,1.990231e-07,3.971570e-20,19.401038
2,9.020258,3.350149e-18,17.474936,KEGG_ADHERENS_JUNCTION,2.715652e-07,2.715652e-07,4.022216e-18,17.395535
3,12.187231,2.531560e-30,29.596612,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,2.138752e-07,2.138752e-07,7.091960e-30,29.149234
4,12.660265,2.570743e-32,31.589941,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,2.471272e-07,2.471272e-07,1.057754e-31,30.975615
...,...,...,...,...,...,...,...,...
390,13.003385,8.694028e-34,33.060779,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,2.473844e-07,2.473844e-07,6.243893e-33,32.204545
391,9.963778,1.448963e-21,20.838943,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,1.097209e-07,1.097209e-07,2.022404e-21,20.694132
392,13.309182,4.088855e-35,34.388398,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,1.135037e-07,1.135037e-07,6.460391e-34,33.189741
393,10.789378,1.087196e-24,23.963692,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,2.278621e-07,2.278621e-07,1.849709e-24,23.732897


In [217]:
common_pathway_pval_df.to_csv(f"[{date}_{num}]{asthma}_Common_Pathways_p-values_FDR_Correction.csv", header = True, index = False)

---

In [218]:
common_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)

,statistic,common_pval,-log10(common_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
28,13.559038,3.278150e-36,35.484371,KEGG_B_CELL_RECEPTOR_SIGNALING_PATHWAY,3.066794e-07,3.066794e-07,2.947234e-34,33.530585
378,13.539997,3.976441e-36,35.400505,KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_...,2.438517e-07,2.438517e-07,2.947234e-34,33.530585
337,13.560176,3.240495e-36,35.489389,KEGG_MEDICUS_REFERENCE_TYPE_II_INTERFERON_TO_J...,3.108042e-07,3.108042e-07,2.947234e-34,33.530585
59,13.520391,4.850554e-36,35.314209,KEGG_GLIOMA,2.277667e-07,2.277667e-07,2.947234e-34,33.530585
63,13.662245,1.148332e-36,35.939932,KEGG_GLYCINE_SERINE_AND_THREONINE_METABOLISM,3.608884e-07,3.608884e-07,2.947234e-34,33.530585
...,...,...,...,...,...,...,...,...
89,6.935437,1.172470e-11,10.930898,KEGG_LYSOSOME,3.304598e-07,3.304598e-07,1.184465e-11,10.926478
274,6.932091,1.198206e-11,10.921469,KEGG_MEDICUS_REFERENCE_IL1_IL1R_JNK_SIGNALING_...,1.899282e-07,1.899282e-07,1.207376e-11,10.918158
267,6.872003,1.767041e-11,10.752753,KEGG_MEDICUS_REFERENCE_GPCR_PLCB_ITPR_SIGNALIN...,2.085963e-07,2.085963e-07,1.776034e-11,10.750549
210,6.666554,6.534320e-11,10.184800,KEGG_MEDICUS_REFERENCE_ARL8_REGULATED_MICROTUB...,1.477453e-07,1.477453e-07,6.550904e-11,10.183699


In [219]:
male_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)

,statistic,male_pval,-log10(male_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
235,7.556537,1.351763e-12,11.869099,KEGG_MEDICUS_REFERENCE_CXCR4_GNB_G_PLCB_PKC_SI...,1.501074e-07,1.501074e-07,3.755520e-11,10.425330
241,7.460352,2.404798e-12,11.618921,KEGG_MEDICUS_REFERENCE_DISASSEMBLY_OF_MCC,1.812037e-07,1.812037e-07,3.755520e-11,10.425330
338,7.786801,3.350724e-13,12.474861,KEGG_MEDICUS_REFERENCE_TYPE_I_IFN_SIGNALING_PA...,1.190043e-07,1.190043e-07,3.755520e-11,10.425330
160,7.636421,8.352695e-13,12.078173,KEGG_TRYPTOPHAN_METABOLISM,7.015004e-08,7.015004e-08,3.755520e-11,10.425330
154,7.507587,1.813148e-12,11.741567,KEGG_SYSTEMIC_LUPUS_ERYTHEMATOSUS,8.026146e-08,8.026146e-08,3.755520e-11,10.425330
...,...,...,...,...,...,...,...,...
326,3.364807,9.142680e-04,3.038926,KEGG_MEDICUS_REFERENCE_TLR3_IRF7_SIGNALING_PAT...,6.955339e-08,6.955339e-08,9.236212e-04,3.034506
214,3.103847,2.179935e-03,2.661557,KEGG_MEDICUS_REFERENCE_AUTOPHAGY_VESICLE_NUCLE...,6.257406e-08,6.257406e-08,2.196618e-03,2.658245
285,3.103019,2.185764e-03,2.660397,KEGG_MEDICUS_REFERENCE_KINETOCHORE_FIBER_ORGAN...,5.059141e-08,5.059141e-08,2.196887e-03,2.658192
246,3.098675,2.216592e-03,2.654314,KEGG_MEDICUS_REFERENCE_DYNEIN_RECRUITMENT_TO_T...,6.685320e-08,6.685320e-08,2.222217e-03,2.653213


In [220]:
female_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)

,statistic,female_pval,-log10(female_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
266,10.583468,1.008852e-22,21.996172,KEGG_MEDICUS_REFERENCE_GPCR_PI3K_SIGNALING_PAT...,1.881042e-07,1.881042e-07,5.380482e-21,20.269179
199,10.623185,7.350699e-23,22.133671,KEGG_MEDICUS_PATHOGEN_SARS_COV_2_NSP1_TO_TRANS...,1.117766e-07,1.117766e-07,5.380482e-21,20.269179
386,10.609642,8.189183e-23,22.086759,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH...,1.278838e-07,1.278838e-07,5.380482e-21,20.269179
203,10.633348,6.778091e-23,22.168893,KEGG_MEDICUS_REFERENCE_ACTH_CORTISOL_SIGNALING...,1.248720e-07,1.248720e-07,5.380482e-21,20.269179
295,10.559767,1.218320e-22,21.914239,KEGG_MEDICUS_REFERENCE_MICROTUBULE_NUCLEATION,1.679912e-07,1.679912e-07,5.380482e-21,20.269179
...,...,...,...,...,...,...,...,...
270,4.389506,1.532745e-05,4.814530,KEGG_MEDICUS_REFERENCE_HOMOLOGOUS_RECOMBINATIO...,5.262938e-08,5.262938e-08,1.548425e-05,4.810110
30,4.055810,6.243573e-05,4.204567,KEGG_CARDIAC_MUSCLE_CONTRACTION,5.158699e-08,5.158699e-08,6.291355e-05,4.201256
24,3.959849,9.195847e-05,4.036408,KEGG_BETA_ALANINE_METABOLISM,4.533970e-08,4.533970e-08,9.242646e-05,4.034204
380,3.276941,1.161367e-03,2.935030,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_ATP1...,4.434464e-08,4.434464e-08,1.164315e-03,2.933929


In [221]:
common_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:15].values

array(['KEGG_B_CELL_RECEPTOR_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_TDP43_TO_ELECTRON_TRANSFER_IN_COMPLEX_I',
       'KEGG_MEDICUS_REFERENCE_TYPE_II_INTERFERON_TO_JAK_STAT_SIGNALING_PATHWAY',
       'KEGG_GLIOMA', 'KEGG_GLYCINE_SERINE_AND_THREONINE_METABOLISM',
       'KEGG_MEDICUS_REFERENCE_GNRH_GNRHR_PLCB_PKC_SIGNALING_PATHWAY',
       'KEGG_INSULIN_SIGNALING_PATHWAY', 'KEGG_NITROGEN_METABOLISM',
       'KEGG_PYRIMIDINE_METABOLISM', 'KEGG_FOCAL_ADHESION',
       'KEGG_MEDICUS_REFERENCE_TYPE_I_INTERFERON_TO_JAK_STAT_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_DNA_END_RESECTION_AND_RPA_LOADING',
       'KEGG_MEDICUS_REFERENCE_N_GLYCAN_BIOSYNTHESIS',
       'KEGG_AUTOIMMUNE_THYROID_DISEASE',
       'KEGG_MEDICUS_REFERENCE_ITGA_B_FAK_CDC42_SIGNALING_PATHWAY'],
      dtype=object)

In [222]:
male_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:15].values

array(['KEGG_MEDICUS_REFERENCE_CXCR4_GNB_G_PLCB_PKC_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_DISASSEMBLY_OF_MCC',
       'KEGG_MEDICUS_REFERENCE_TYPE_I_IFN_SIGNALING_PATHWAY',
       'KEGG_TRYPTOPHAN_METABOLISM', 'KEGG_SYSTEMIC_LUPUS_ERYTHEMATOSUS',
       'KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_ATP2B3_TO_ANGIOTENSIN_ALDOSTERONE_SIGNALING_PATHWAY',
       'KEGG_TGF_BETA_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_ENV_FACTOR_IRON_TO_ANTEROGRADE_AXONAL_TRANSPORT',
       'KEGG_MEDICUS_REFERENCE_GLYCOLYSIS',
       'KEGG_MEDICUS_REFERENCE_IL6_FAMILY_TO_JAK_STAT_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_AGE_RAGE_SIGNALING_PATHWAY',
       'KEGG_BIOSYNTHESIS_OF_UNSATURATED_FATTY_ACIDS',
       'KEGG_CYSTEINE_AND_METHIONINE_METABOLISM',
       'KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM',
       'KEGG_MEDICUS_REFERENCE_RAB7_REGULATED_MICROTUBULE_MINUS_END_DIRECTED_TRANSPORT'],
      dtype=object)

In [223]:
female_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:15].values

array(['KEGG_MEDICUS_REFERENCE_GPCR_PI3K_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_PATHOGEN_SARS_COV_2_NSP1_TO_TRANSLATION_INITIATION',
       'KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH1_TO_HEDGEHOG_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_ACTH_CORTISOL_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_MICROTUBULE_NUCLEATION',
       'KEGG_GLYCOSAMINOGLYCAN_BIOSYNTHESIS_HEPARAN_SULFATE',
       'KEGG_MEDICUS_REFERENCE_ELECTRON_TRANSFER_IN_COMPLEX_I',
       'KEGG_PPAR_SIGNALING_PATHWAY', 'KEGG_CHRONIC_MYELOID_LEUKEMIA',
       'KEGG_MEDICUS_REFERENCE_GF_RTK_RAS_PI3K_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_ENV_FACTOR_DCE_TO_DNA_ADDUCTS', 'KEGG_ENDOCYTOSIS',
       'KEGG_MEDICUS_REFERENCE_CXCL12_CXCR4_PKC_ERK_SIGNALING_PATHAWAY',
       'KEGG_MEDICUS_REFERENCE_ACTIVATION_OF_PRC2.2_BY_UBIQUITINATION_OF_H2AK119_IN_GERMLINE_GENES',
       'KEGG_MEDICUS_REFERENCE_CYTOKINE_JAK_STAT_SIGNALING_PATHWAY'],
      dtype=object)

In [224]:
np.intersect1d(common_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:10].values, male_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:10].values)

array([], dtype=object)

In [225]:
np.intersect1d(common_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:10].values, female_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:10].values)

array([], dtype=object)

In [226]:
np.intersect1d(male_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:10].values, female_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:10].values)

array([], dtype=object)

In [227]:
np.intersect1d(common_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:15].values, male_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:15].values)

array([], dtype=object)

In [228]:
np.intersect1d(common_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:15].values, female_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:15].values)

array([], dtype=object)

In [229]:
np.intersect1d(male_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:15].values, female_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:15].values)

array([], dtype=object)

---

In [ ]:
pd.set_option("display.max_rows", None)

In [230]:
significant_cmn_pathway_df = cmn_pathway_df[['KEGG_B_CELL_RECEPTOR_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_TDP43_TO_ELECTRON_TRANSFER_IN_COMPLEX_I',
       'KEGG_MEDICUS_REFERENCE_TYPE_II_INTERFERON_TO_JAK_STAT_SIGNALING_PATHWAY',
       'KEGG_GLIOMA', 'KEGG_GLYCINE_SERINE_AND_THREONINE_METABOLISM',
       'KEGG_MEDICUS_REFERENCE_GNRH_GNRHR_PLCB_PKC_SIGNALING_PATHWAY',
       'KEGG_INSULIN_SIGNALING_PATHWAY', 'KEGG_NITROGEN_METABOLISM',
       'KEGG_PYRIMIDINE_METABOLISM', 'KEGG_FOCAL_ADHESION',
       'KEGG_MEDICUS_REFERENCE_TYPE_I_INTERFERON_TO_JAK_STAT_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_DNA_END_RESECTION_AND_RPA_LOADING',
       'KEGG_MEDICUS_REFERENCE_N_GLYCAN_BIOSYNTHESIS',
       'KEGG_AUTOIMMUNE_THYROID_DISEASE',
       'KEGG_MEDICUS_REFERENCE_ITGA_B_FAK_CDC42_SIGNALING_PATHWAY']]
significant_cmn_pathway_df

,KEGG_B_CELL_RECEPTOR_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_TDP43_TO_ELECTRON_TRANSFER_IN_COMPLEX_I,KEGG_MEDICUS_REFERENCE_TYPE_II_INTERFERON_TO_JAK_STAT_SIGNALING_PATHWAY,KEGG_GLIOMA,KEGG_GLYCINE_SERINE_AND_THREONINE_METABOLISM,KEGG_MEDICUS_REFERENCE_GNRH_GNRHR_PLCB_PKC_SIGNALING_PATHWAY,KEGG_INSULIN_SIGNALING_PATHWAY,KEGG_NITROGEN_METABOLISM,KEGG_PYRIMIDINE_METABOLISM,KEGG_FOCAL_ADHESION,KEGG_MEDICUS_REFERENCE_TYPE_I_INTERFERON_TO_JAK_STAT_SIGNALING_PATHWAY,KEGG_MEDICUS_REFERENCE_DNA_END_RESECTION_AND_RPA_LOADING,KEGG_MEDICUS_REFERENCE_N_GLYCAN_BIOSYNTHESIS,KEGG_AUTOIMMUNE_THYROID_DISEASE,KEGG_MEDICUS_REFERENCE_ITGA_B_FAK_CDC42_SIGNALING_PATHWAY
0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2,-1.164010e-06,1.014399e-06,1.162041e-06,-8.733826e-07,-1.376529e-06,9.066936e-07,-1.363946e-06,-1.438333e-06,1.400231e-06,-8.420934e-07,-9.401028e-07,1.085209e-06,8.067537e-07,1.153733e-06,9.193490e-07
3,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
4,7.623888e-07,-5.049927e-07,-1.109376e-06,6.906629e-07,-1.110463e-06,-1.277275e-06,1.070665e-06,3.398297e-07,-6.839602e-07,-1.088025e-06,4.199970e-07,1.460255e-07,5.384105e-07,-1.399792e-06,8.184900e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
532,2.789710e-07,-5.840490e-07,6.224736e-07,3.807234e-07,5.169149e-07,-4.182006e-07,3.705111e-07,9.229795e-07,-9.748428e-07,-7.160032e-07,-1.145421e-06,-1.296520e-06,8.949121e-08,-7.591331e-07,-3.437161e-07
533,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
534,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [231]:
np.sign(significant_cmn_pathway_df).apply(pd.Series.value_counts).T

,-1.0,0.0,1.0
KEGG_B_CELL_RECEPTOR_SIGNALING_PATHWAY,131,370,35
KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_TDP43_TO_ELECTRON_TRANSFER_IN_COMPLEX_I,47,370,119
KEGG_MEDICUS_REFERENCE_TYPE_II_INTERFERON_TO_JAK_STAT_SIGNALING_PATHWAY,28,370,138
KEGG_GLIOMA,102,370,64
KEGG_GLYCINE_SERINE_AND_THREONINE_METABOLISM,135,370,31
...,...,...,...
KEGG_MEDICUS_REFERENCE_TYPE_I_INTERFERON_TO_JAK_STAT_SIGNALING_PATHWAY,131,370,35
KEGG_MEDICUS_REFERENCE_DNA_END_RESECTION_AND_RPA_LOADING,40,370,126
KEGG_MEDICUS_REFERENCE_N_GLYCAN_BIOSYNTHESIS,17,370,149
KEGG_AUTOIMMUNE_THYROID_DISEASE,22,370,144


In [232]:
significant_male_pathway_df = male_pathway_df[['KEGG_MEDICUS_REFERENCE_CXCR4_GNB_G_PLCB_PKC_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_DISASSEMBLY_OF_MCC',
       'KEGG_MEDICUS_REFERENCE_TYPE_I_IFN_SIGNALING_PATHWAY',
       'KEGG_TRYPTOPHAN_METABOLISM', 'KEGG_SYSTEMIC_LUPUS_ERYTHEMATOSUS',
       'KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_ATP2B3_TO_ANGIOTENSIN_ALDOSTERONE_SIGNALING_PATHWAY',
       'KEGG_TGF_BETA_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_ENV_FACTOR_IRON_TO_ANTEROGRADE_AXONAL_TRANSPORT',
       'KEGG_MEDICUS_REFERENCE_GLYCOLYSIS',
       'KEGG_MEDICUS_REFERENCE_IL6_FAMILY_TO_JAK_STAT_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_AGE_RAGE_SIGNALING_PATHWAY',
       'KEGG_BIOSYNTHESIS_OF_UNSATURATED_FATTY_ACIDS',
       'KEGG_CYSTEINE_AND_METHIONINE_METABOLISM',
       'KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM',
       'KEGG_MEDICUS_REFERENCE_RAB7_REGULATED_MICROTUBULE_MINUS_END_DIRECTED_TRANSPORT']]
significant_male_pathway_df

,KEGG_MEDICUS_REFERENCE_CXCR4_GNB_G_PLCB_PKC_SIGNALING_PATHWAY,KEGG_MEDICUS_REFERENCE_DISASSEMBLY_OF_MCC,KEGG_MEDICUS_REFERENCE_TYPE_I_IFN_SIGNALING_PATHWAY,KEGG_TRYPTOPHAN_METABOLISM,KEGG_SYSTEMIC_LUPUS_ERYTHEMATOSUS,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_ATP2B3_TO_ANGIOTENSIN_ALDOSTERONE_SIGNALING_PATHWAY,KEGG_TGF_BETA_SIGNALING_PATHWAY,KEGG_MEDICUS_ENV_FACTOR_IRON_TO_ANTEROGRADE_AXONAL_TRANSPORT,KEGG_MEDICUS_REFERENCE_GLYCOLYSIS,KEGG_MEDICUS_REFERENCE_IL6_FAMILY_TO_JAK_STAT_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_AGE_RAGE_SIGNALING_PATHWAY,KEGG_BIOSYNTHESIS_OF_UNSATURATED_FATTY_ACIDS,KEGG_CYSTEINE_AND_METHIONINE_METABOLISM,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,KEGG_MEDICUS_REFERENCE_RAB7_REGULATED_MICROTUBULE_MINUS_END_DIRECTED_TRANSPORT
0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2,-5.403821e-07,-4.943189e-07,-2.655778e-07,-2.401328e-07,3.086177e-07,-2.567941e-07,-8.126387e-07,4.052316e-07,-9.191825e-07,-4.363890e-07,5.714267e-07,-4.230047e-07,6.034501e-07,-8.129619e-07,-5.696217e-07
3,8.754002e-07,-6.782291e-07,6.705444e-07,-2.015375e-07,-3.863494e-07,-1.489973e-07,5.938400e-07,1.790222e-07,5.979237e-07,-3.037415e-07,4.237775e-07,-4.716135e-07,5.180000e-07,-3.450770e-07,-5.165271e-07
4,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
202,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
203,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
204,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [233]:
np.sign(significant_male_pathway_df).apply(pd.Series.value_counts).T

,-1.0,0.0,1.0
KEGG_MEDICUS_REFERENCE_CXCR4_GNB_G_PLCB_PKC_SIGNALING_PATHWAY,22,151,33
KEGG_MEDICUS_REFERENCE_DISASSEMBLY_OF_MCC,54,151,1
KEGG_MEDICUS_REFERENCE_TYPE_I_IFN_SIGNALING_PATHWAY,22,151,33
KEGG_TRYPTOPHAN_METABOLISM,52,151,3
KEGG_SYSTEMIC_LUPUS_ERYTHEMATOSUS,30,151,25
...,...,...,...
KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_AGE_RAGE_SIGNALING_PATHWAY,5,151,50
KEGG_BIOSYNTHESIS_OF_UNSATURATED_FATTY_ACIDS,52,151,3
KEGG_CYSTEINE_AND_METHIONINE_METABOLISM,5,151,50
KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,53,151,2


In [234]:
significant_female_pathway_df = female_pathway_df[['KEGG_MEDICUS_REFERENCE_GPCR_PI3K_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_PATHOGEN_SARS_COV_2_NSP1_TO_TRANSLATION_INITIATION',
       'KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH1_TO_HEDGEHOG_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_ACTH_CORTISOL_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_MICROTUBULE_NUCLEATION',
       'KEGG_GLYCOSAMINOGLYCAN_BIOSYNTHESIS_HEPARAN_SULFATE',
       'KEGG_MEDICUS_REFERENCE_ELECTRON_TRANSFER_IN_COMPLEX_I',
       'KEGG_PPAR_SIGNALING_PATHWAY', 'KEGG_CHRONIC_MYELOID_LEUKEMIA',
       'KEGG_MEDICUS_REFERENCE_GF_RTK_RAS_PI3K_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_ENV_FACTOR_DCE_TO_DNA_ADDUCTS', 'KEGG_ENDOCYTOSIS',
       'KEGG_MEDICUS_REFERENCE_CXCL12_CXCR4_PKC_ERK_SIGNALING_PATHAWAY',
       'KEGG_MEDICUS_REFERENCE_ACTIVATION_OF_PRC2.2_BY_UBIQUITINATION_OF_H2AK119_IN_GERMLINE_GENES',
       'KEGG_MEDICUS_REFERENCE_CYTOKINE_JAK_STAT_SIGNALING_PATHWAY']]
significant_female_pathway_df

,KEGG_MEDICUS_REFERENCE_GPCR_PI3K_SIGNALING_PATHWAY,KEGG_MEDICUS_PATHOGEN_SARS_COV_2_NSP1_TO_TRANSLATION_INITIATION,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH1_TO_HEDGEHOG_SIGNALING_PATHWAY,KEGG_MEDICUS_REFERENCE_ACTH_CORTISOL_SIGNALING_PATHWAY,KEGG_MEDICUS_REFERENCE_MICROTUBULE_NUCLEATION,KEGG_GLYCOSAMINOGLYCAN_BIOSYNTHESIS_HEPARAN_SULFATE,KEGG_MEDICUS_REFERENCE_ELECTRON_TRANSFER_IN_COMPLEX_I,KEGG_PPAR_SIGNALING_PATHWAY,KEGG_CHRONIC_MYELOID_LEUKEMIA,KEGG_MEDICUS_REFERENCE_GF_RTK_RAS_PI3K_SIGNALING_PATHWAY,KEGG_MEDICUS_ENV_FACTOR_DCE_TO_DNA_ADDUCTS,KEGG_ENDOCYTOSIS,KEGG_MEDICUS_REFERENCE_CXCL12_CXCR4_PKC_ERK_SIGNALING_PATHAWAY,KEGG_MEDICUS_REFERENCE_ACTIVATION_OF_PRC2.2_BY_UBIQUITINATION_OF_H2AK119_IN_GERMLINE_GENES,KEGG_MEDICUS_REFERENCE_CYTOKINE_JAK_STAT_SIGNALING_PATHWAY
0,-2.527132e-07,-2.699284e-07,-1.827904e-07,-1.546697e-07,-4.942669e-07,2.027028e-07,-3.257127e-07,2.731188e-07,-1.646970e-07,-7.104701e-08,-8.607122e-08,5.311403e-07,-1.329558e-08,4.207398e-07,-9.428952e-08
1,-6.845460e-07,-5.847912e-07,4.494804e-07,5.680517e-07,-8.440014e-07,3.162728e-07,-7.270641e-07,7.533906e-07,-5.159622e-07,8.093705e-07,5.978118e-07,5.015960e-07,4.153896e-07,-1.060937e-06,-3.264512e-07
2,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
3,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
4,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
326,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
327,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
328,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [235]:
np.sign(significant_female_pathway_df).apply(pd.Series.value_counts).T

,-1.0,0.0,1.0
KEGG_MEDICUS_REFERENCE_GPCR_PI3K_SIGNALING_PATHWAY,41,225,64
KEGG_MEDICUS_PATHOGEN_SARS_COV_2_NSP1_TO_TRANSLATION_INITIATION,50,225,55
KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH1_TO_HEDGEHOG_SIGNALING_PATHWAY,8,225,97
KEGG_MEDICUS_REFERENCE_ACTH_CORTISOL_SIGNALING_PATHWAY,62,225,43
KEGG_MEDICUS_REFERENCE_MICROTUBULE_NUCLEATION,50,225,55
...,...,...,...
KEGG_MEDICUS_ENV_FACTOR_DCE_TO_DNA_ADDUCTS,25,225,80
KEGG_ENDOCYTOSIS,62,225,43
KEGG_MEDICUS_REFERENCE_CXCL12_CXCR4_PKC_ERK_SIGNALING_PATHAWAY,18,225,87
KEGG_MEDICUS_REFERENCE_ACTIVATION_OF_PRC2.2_BY_UBIQUITINATION_OF_H2AK119_IN_GERMLINE_GENES,80,225,25


---

---

---

In [12]:
common_gene_pval_df = pd.read_csv(f"[0911_1]{asthma}_Common_Genes_p-values_FDR_Correction.csv")
male_gene_pval_df = pd.read_csv(f"[0911_1]{asthma}_Male_Genes_p-values_FDR_Correction.csv")
female_gene_pval_df = pd.read_csv(f"[0911_1]{asthma}_Female_Genes_p-values_FDR_Correction.csv")

In [9]:
common_gene_pval_df

,statistic,common_pval,-log10(common_pval),Gene,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
0,6.079989,2.287627e-09,8.640615,DPM1,6.470109e-10,6.470109e-10,2.520030e-09,8.598594
1,9.380847,1.848442e-19,18.733194,FGR,2.853710e-09,2.853710e-09,5.911863e-19,18.228276
2,7.258475,1.386100e-12,11.858206,CFH,5.350517e-10,5.350517e-10,1.817968e-12,11.740414
3,7.604117,1.297338e-13,12.886947,FUCA2,7.120090e-11,7.120090e-11,1.835856e-13,12.736161
4,8.005210,7.468135e-15,14.126788,GCLC,1.229822e-12,1.229822e-12,1.192993e-14,13.923362
...,...,...,...,...,...,...,...,...
4495,11.189845,2.913395e-26,25.535601,H3C2,1.318210e-09,1.318210e-09,7.757559e-25,24.110275
4496,9.035209,2.975707e-18,17.526410,H3C3,7.361822e-10,7.361822e-10,7.544046e-18,17.122396
4497,5.207112,2.740145e-07,6.562226,NPBWR1,3.084771e-10,3.084771e-10,2.835936e-07,6.547304
4498,10.378476,4.088994e-23,22.388383,UGT1A3,3.823638e-09,3.823638e-09,3.036382e-22,21.517644


In [11]:
common_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)[:30]

,statistic,common_pval,-log10(common_pval),Gene,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
135,12.404758,3.103402e-31,30.508162,IGF1,3.370056e-09,3.370056e-09,1.161412e-27,26.935014
389,12.352220,5.161831e-31,30.287196,NCK2,3.132305e-09,3.132305e-09,1.161412e-27,26.935014
344,12.191960,2.419178e-30,29.616332,TNFRSF1A,4.439549e-09,4.439549e-09,2.721575e-27,26.565180
3364,12.215020,1.938341e-30,29.712570,IRS1,3.636946e-09,3.636946e-09,2.721575e-27,26.565180
817,12.152969,3.516872e-30,29.453843,PCK2,3.179476e-09,3.179476e-09,3.165185e-27,26.499601
...,...,...,...,...,...,...,...,...
98,11.806384,9.498164e-29,28.022360,FYN,2.984844e-09,2.984844e-09,1.694051e-26,25.771074
1628,11.792066,1.087103e-28,27.963729,APAF1,4.353013e-09,4.353013e-09,1.811839e-26,25.741880
3315,11.773216,1.298360e-28,27.886605,GNG4,2.708483e-09,2.708483e-09,2.014697e-26,25.695790
4002,11.773786,1.291414e-28,27.888935,WNT7B,3.756146e-09,3.756146e-09,2.014697e-26,25.695790


In [13]:
drop_log_common_gene_pval_df = common_gene_pval_df.drop(columns=['-log10(common_pval)', '-log10(FDR_BH)'])
drop_log_male_gene_pval_df = male_gene_pval_df.drop(columns=['-log10(male_pval)', '-log10(FDR_BH)'])
drop_log_female_gene_pval_df = female_gene_pval_df.drop(columns=['-log10(female_pval)', '-log10(FDR_BH)'])

In [15]:
drop_log_common_gene_pval_df.sort_values(by='FDR_BH', ascending=True)[:30]

,statistic,common_pval,Gene,ABS_Importance,Importance,FDR_BH
389,12.352220,5.161831e-31,NCK2,3.132305e-09,3.132305e-09,1.161412e-27
135,12.404758,3.103402e-31,IGF1,3.370056e-09,3.370056e-09,1.161412e-27
344,12.191960,2.419178e-30,TNFRSF1A,4.439549e-09,4.439549e-09,2.721575e-27
3364,12.215020,1.938341e-30,IRS1,3.636946e-09,3.636946e-09,2.721575e-27
817,12.152969,3.516872e-30,PCK2,3.179476e-09,3.179476e-09,3.165185e-27
...,...,...,...,...,...,...
98,11.806384,9.498164e-29,FYN,2.984844e-09,2.984844e-09,1.694051e-26
1628,11.792066,1.087103e-28,APAF1,4.353013e-09,4.353013e-09,1.811839e-26
3315,11.773216,1.298360e-28,GNG4,2.708483e-09,2.708483e-09,2.014697e-26
4002,11.773786,1.291414e-28,WNT7B,3.756146e-09,3.756146e-09,2.014697e-26


In [16]:
drop_log_common_gene_pval_df.sort_values(by='FDR_BH', ascending=True)[:30].sort_values(by='ABS_Importance', ascending=False)

,statistic,common_pval,Gene,ABS_Importance,Importance,FDR_BH
1457,11.941499,2.644604e-29,SOS1,1.069349e-08,1.069349e-08,6.611510e-27
1040,12.035554,1.080732e-29,PIK3CG,1.058299e-08,1.058299e-08,5.561384e-27
2855,11.991595,1.642765e-29,CREB3L1,6.346730e-09,6.346730e-09,6.160367e-27
1048,11.910672,3.542941e-29,MET,5.361075e-09,5.361075e-09,7.246924e-27
2145,11.758801,1.487058e-28,WNT10A,5.314865e-09,5.314865e-09,2.177976e-26
...,...,...,...,...,...,...
2250,11.900066,3.917577e-29,PNPT1,1.569774e-09,1.569774e-09,7.664825e-27
3814,11.944456,2.571392e-29,CACNB4,1.524165e-09,1.524165e-09,6.611510e-27
151,11.981514,1.808118e-29,BIRC3,1.353717e-09,1.353717e-09,6.258870e-27
1986,12.084656,6.763057e-30,PRKAA1,1.046044e-09,1.046044e-09,4.347680e-27


In [17]:
top_common_gene_pval_df = drop_log_common_gene_pval_df.sort_values(
    by=['FDR_BH', 'statistic'], 
    ascending=[True, False]
)[:30].sort_values(
    by='ABS_Importance', 
    ascending=False
).reset_index(drop=True)

In [18]:
top_male_gene_pval_df = drop_log_male_gene_pval_df.sort_values(
    by=['FDR_BH', 'statistic'], 
    ascending=[True, False]
)[:30].sort_values(
    by='ABS_Importance', 
    ascending=False
).reset_index(drop=True)

In [19]:
top_female_gene_pval_df = drop_log_female_gene_pval_df.sort_values(
    by=['FDR_BH', 'statistic'], 
    ascending=[True, False]
)[:30].sort_values(
    by='ABS_Importance', 
    ascending=False
).reset_index(drop=True)

In [23]:
top_common_gene_pval_df['Gene'].values

array(['SOS1', 'PIK3CG', 'CREB3L1', 'MET', 'WNT10A', 'PPP3CC', 'TRAF6',
       'TNFRSF1A', 'UGT1A8', 'APAF1', 'POLD1', 'SHC2', 'WNT7B', 'IRS1',
       'IGF1', 'IRAK1', 'PCK2', 'NCK2', 'ESPL1', 'POLA1', 'PPP2R1A',
       'MGST1', 'FYN', 'GNG4', 'PRKAA2', 'PNPT1', 'CACNB4', 'BIRC3',
       'PRKAA1', 'WIPF3'], dtype=object)

In [21]:
top_male_gene_pval_df

,statistic,male_pval,Gene,ABS_Importance,Importance,FDR_BH
0,6.955572,4.620797e-11,ALDH1B1,4.358375e-09,4.358375e-09,9.040689e-09
1,7.633609,8.495851e-13,WNT9A,3.724897e-09,3.724897e-09,3.823133e-09
2,7.089103,2.138429e-11,ITGA7,3.538452e-09,3.538452e-09,7.402253e-09
3,6.956115,4.606420e-11,ITGA10,3.403393e-09,3.403393e-09,9.040689e-09
4,7.148619,1.512801e-11,PTCH1,3.260647e-09,3.260647e-09,7.402253e-09
...,...,...,...,...,...,...
25,6.911004,5.964599e-11,MCM2,8.048668e-10,8.048668e-10,9.585963e-09
26,6.894110,6.568972e-11,DNAH11,8.028705e-10,8.028705e-10,9.853458e-09
27,7.023966,3.117336e-11,RFC3,7.780455e-10,7.780455e-10,8.901934e-09
28,7.075081,2.319565e-11,GSTM3,7.696911e-10,7.696911e-10,7.455746e-09


In [22]:
top_female_gene_pval_df

,statistic,female_pval,Gene,ABS_Importance,Importance,FDR_BH
0,9.751060,6.703273e-20,PIK3CB,6.081607e-09,6.081607e-09,6.892944e-17
1,9.656465,1.378589e-19,PIK3CA,3.740809e-09,3.740809e-09,6.892944e-17
2,9.746548,6.938416e-20,PIK3CD,3.652380e-09,3.652380e-09,6.892944e-17
3,9.339231,1.504644e-18,DVL1,3.288760e-09,3.288760e-09,2.334793e-16
4,9.539324,3.349156e-19,TUBB8,2.990536e-09,2.990536e-09,1.076514e-16
...,...,...,...,...,...,...
25,9.352410,1.363622e-18,EFNA4,1.271126e-09,1.271126e-09,2.321217e-16
26,9.667827,1.264472e-19,LPL,1.016471e-09,1.016471e-09,6.892944e-17
27,9.348337,1.405753e-18,MAN1B1,7.147426e-10,7.147426e-10,2.321217e-16
28,9.404180,9.257133e-19,PRKAB2,5.074738e-10,5.074738e-10,1.983671e-16


In [24]:
top_common_gene_pval_df.to_csv(f"{asthma}_Significant_Common_Genes_p-values_FDR_Correction.csv", header = True, index = False)
top_male_gene_pval_df.to_csv(f"{asthma}_Significant_Male_Genes_p-values_FDR_Correction.csv", header = True, index = False)
top_female_gene_pval_df.to_csv(f"{asthma}_Significant_Female_Genes_p-values_FDR_Correction.csv", header = True, index = False)

---

---

In [25]:
common_pathway_pval_df = pd.read_csv(f"[0911_1]{asthma}_Common_Pathways_p-values_FDR_Correction.csv")
male_pathway_pval_df = pd.read_csv(f"[0911_1]{asthma}_Male_Pathways_p-values_FDR_Correction.csv")
female_pathway_pval_df = pd.read_csv(f"[0911_1]{asthma}_Female_Pathways_p-values_FDR_Correction.csv")

In [26]:
drop_log_common_pathway_pval_df = common_pathway_pval_df.drop(columns=['-log10(common_pval)', '-log10(FDR_BH)'])
drop_log_male_pathway_pval_df = male_pathway_pval_df.drop(columns=['-log10(male_pval)', '-log10(FDR_BH)'])
drop_log_female_pathway_pval_df = female_pathway_pval_df.drop(columns=['-log10(female_pval)', '-log10(FDR_BH)'])

In [27]:
top_common_pathway_pval_df = drop_log_common_pathway_pval_df.sort_values(
    by=['FDR_BH', 'statistic'], 
    ascending=[True, False]
)[:30].sort_values(
    by='ABS_Importance', 
    ascending=False
).reset_index(drop=True)

In [28]:
top_male_pathway_pval_df = drop_log_male_pathway_pval_df.sort_values(
    by=['FDR_BH', 'statistic'], 
    ascending=[True, False]
)[:30].sort_values(
    by='ABS_Importance', 
    ascending=False
).reset_index(drop=True)

In [29]:
top_female_pathway_pval_df = drop_log_female_pathway_pval_df.sort_values(
    by=['FDR_BH', 'statistic'], 
    ascending=[True, False]
)[:30].sort_values(
    by='ABS_Importance', 
    ascending=False
).reset_index(drop=True)

In [30]:
top_common_pathway_pval_df.to_csv(f"{asthma}_Significant_Common_Pathways_p-values_FDR_Correction.csv", header = True, index = False)
top_male_pathway_pval_df.to_csv(f"{asthma}_Significant_Male_Pathway_p-values_FDR_Correction.csv", header = True, index = False)
top_female_pathway_pval_df.to_csv(f"{asthma}_Significant_Female_Pathway_p-values_FDR_Correction.csv", header = True, index = False)